# Replicación · ¿Diversidad, familiaridad o liderazgo?
**Diseño de equipos científicos e impacto futuro de la investigación en la interfaz entre orientación emprendedora y orientación al mercado**

Este cuaderno contiene todo el flujo de análisis del artículo, desde las exportaciones de Web of Science y Scopus hasta las tablas y figuras finales. Se ejecuta de arriba abajo en Google Colab y guarda los resultados intermedios en Drive.

| Parte | Qué hace | Entrada principal | Salida principal | Tiempo aprox. |
|---|---|---|---|---|
| **1. BibFusion** | Integra WoS y Scopus y completa autores y referencias con OpenAlex | `scopus.csv`, `wos.txt`, `scimago.csv` | tablas `All_*.csv` | horas |
| **2. Corpus y equipos** | Quita duplicados, unifica la identidad de cada autor | `All_*.csv` | `corpus_principal_442.csv`, `equipos_personas_por_articulo.csv` | minutos |
| **3. Referencias** | Cruza las referencias citadas por cada artículo | `All_Citation.csv` | `citas_principales_final.csv` | minutos |
| **4. Descriptivos** | Describe el corpus y revisa que los artículos traten de OE y OM | Partes 2 y 3 | `tabla1_descriptiva.csv`, muestra de validación | minutos |
| **5. OpenAlex** | Descarga citas por año, historia previa de los autores y métricas de revista; arma la base de modelos | Parte 2 y clave de OpenAlex | `base_modelo_442.csv` y archivos de OpenAlex | horas |
| **6. Análisis final** | Estima los cinco modelos, genera tablas, figuras y pruebas de robustez | Partes 2 y 5 | `tabla3_final_*.csv`, figuras F2–F7 | minutos |

## Dos formas de reproducir
- **Reproducción rápida (recomendada para revisar los resultados):** copia a Drive, en `MyDrive/BibFusion/trabajo_limpio/`, los archivos de la carpeta `data/` del repositorio, salta a la **Parte 6** y ejecuta sus celdas en orden. No llama a ninguna API y tarda pocos minutos.
- **Reproducción completa:** consigue tus propias exportaciones de WoS y Scopus con las ecuaciones de `search/equations.md` y ejecuta las Partes 1 a 6 en orden. Necesitas una clave de OpenAlex y las Partes 1 y 5 pueden tardar horas.

> Los datos originales de WoS y Scopus no se incluyen en el repositorio por restricciones de licencia.


# BibFusion para estudiantes · Colab
**Versión docente 1.0 — 18 de septiembre de 2026**

Integra exportaciones de Scopus y Web of Science y guarda las siete tablas en tu Drive.
Ejecuta las celdas **1 a 5 en orden** con el botón ▶. No necesitas GPU.

## Antes de comenzar
1. Sube este cuaderno a [Google Colab](https://colab.research.google.com/) (Archivo → Subir cuaderno) y guarda tu copia en Drive.
2. Crea en **Mi unidad** una carpeta `BibFusion/entrada` y coloca:
   - `scopus.csv`: exportación CSV con todos los campos, incluidas referencias.
   - `wos.txt`: texto plano de WoS con **Full Record and Cited References**, formato etiquetado (FN, VR, AU…), no tabla delimitada por tabulaciones.
   - `scimago.csv`: **archivo auxiliar de BibFusion facilitado por el docente**, con `Sourceid`, `Title`, `Issn`, `eIssn`, `year` y `journal_abbr`. No se incluye en este cuaderno y una descarga genérica no lo sustituye.
3. Obtén tu clave en [OpenAlex](https://openalex.org/settings/api). No la escribas en las celdas: el cuaderno te la pedirá de forma oculta.

**Duración:** puede tardar horas porque también consulta referencias citadas. Una ejecución anterior con 196 registros de Scopus y 148 de WoS tardó 4 h 44 min; no es una estimación para tu búsqueda.
Mantén Colab abierto y el computador sin suspensión. Esto no elimina los límites de Colab.

**Recuperación:** guarda resultados de cinco etapas de consultas al completarlas. Si se interrumpe, vuelve a ejecutar las celdas con la misma carpeta y los mismos archivos. Reutilizará esas etapas; **la etapa interrumpida se repite desde su comienzo**. Los puntos de recuperación deben ser los creados por tu propia ejecución; no importes los de otras personas.

**Alcance:** es un cuaderno docente que envuelve una revisión fija de BibFusion; no es una nueva versión oficial. Incluye una corrección documentada de campos nulos en el enriquecimiento de referencias de WoS. No corrige automáticamente DOI mal extraídos, identidades de autores o cobertura bibliográfica. Tener siete archivos no equivale a haber auditado los datos.


In [ ]:
#@title 1. Instalar BibFusion (una vez por sesión)
import subprocess, sys
REVISION = "ffda6561f812a2645c9fb234fa0536a798d694e3"
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    f"git+https://github.com/ladmepaz/bibfusion.git@{REVISION}"
], check=True)
print("Instalación terminada. Continúa con la celda 2.")


In [ ]:
#@title 2. Configurar archivos y conectar Drive
CARPETA = "/content/drive/MyDrive/BibFusion" #@param {type:"string"}
ARCHIVO_SCOPUS = "scopus.csv" #@param {type:"string"}
ARCHIVO_WOS = "wos.txt" #@param {type:"string"}
ARCHIVO_SCIMAGO = "scimago.csv" #@param {type:"string"}
REUTILIZAR_ETAPAS = True #@param {type:"boolean"}

from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
BASE = Path(CARPETA).expanduser()
ENTRADA = BASE / "entrada"
ENTRADA.mkdir(parents=True, exist_ok=True)
print("Carpeta para tus tres archivos:", ENTRADA)
print("Archivos encontrados:", [p.name for p in ENTRADA.iterdir() if p.is_file()])
print("Si falta alguno, súbelo a esa carpeta antes de continuar.")


## 3. Preparación automática
Valida el formato básico, crea copias de trabajo y registra versiones y huellas de los archivos.
Los resultados van a `BibFusion/salidas/ejecucion_…`. Cada conjunto de entradas tiene una carpeta distinta.
Los originales no se modifican. La misma entrada reutiliza su carpeta; no ejecutes dos cuadernos a la vez sobre ella.


In [ ]:
#@title 3. Validar y preparar
import pandas as pd
import hashlib, json, shutil, urllib.request, platform
from datetime import datetime, timezone
from importlib.metadata import version

REVISION = "ffda6561f812a2645c9fb234fa0536a798d694e3"
VERSION_DOCENTE = "1.0"
ARCHIVOS = {"scopus.csv": ENTRADA / ARCHIVO_SCOPUS,
            "wos.txt": ENTRADA / ARCHIVO_WOS,
            "scimago.csv": ENTRADA / ARCHIVO_SCIMAGO}
faltantes = [str(p) for p in ARCHIVOS.values() if not p.is_file()]
if faltantes:
    raise FileNotFoundError("Sube estos archivos y repite esta celda: " + ", ".join(faltantes))
if len({p.resolve() for p in ARCHIVOS.values()}) != 3:
    raise ValueError("Selecciona tres archivos diferentes.")

scopus = pd.read_csv(ARCHIVOS["scopus.csv"], dtype=str, keep_default_na=False)
requeridos = {"Authors", "Author full names", "Title", "Year", "Source title", "References", "DOI", "Affiliations", "Authors with affiliations"}
if not requeridos.issubset(scopus.columns):
    raise ValueError("Scopus: vuelve a exportar CSV con todos los campos. Faltan: " + str(sorted(requeridos - set(scopus.columns))))
if scopus.empty:
    raise ValueError("La exportación de Scopus está vacía.")
wos_texto = ARCHIVOS["wos.txt"].read_text(encoding="utf-8-sig")
lineas = wos_texto.splitlines()
if not (any(s.startswith("FN ") for s in lineas[:5]) and
        any(s.startswith("VR ") for s in lineas[:5]) and
        any(s.strip() == "ER" for s in lineas)):
    raise ValueError("WoS: se necesita texto plano etiquetado con FN, VR y ER.")
if not any(s.startswith("CR ") for s in lineas):
    raise ValueError("WoS no contiene CR. Exporta Full Record and Cited References.")
scimago_cols = set(pd.read_csv(ARCHIVOS["scimago.csv"], nrows=0).columns)
requeridos_sci = {"Sourceid", "Title", "Issn", "eIssn", "year", "journal_abbr"}
if not requeridos_sci.issubset(scimago_cols):
    raise ValueError("Usa el scimago.csv preparado para BibFusion. Faltan: " + str(sorted(requeridos_sci - scimago_cols)))

def huella(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

huellas = {n: huella(p) for n, p in ARCHIVOS.items()}
identidad = {"archivos": huellas, "revision": REVISION, "version_docente": VERSION_DOCENTE}
firma = hashlib.sha256(json.dumps(identidad, sort_keys=True).encode()).hexdigest()
EJECUCION = BASE / "salidas" / ("ejecucion_" + firma[:16])
EJECUCION.mkdir(parents=True, exist_ok=True)
for nombre, origen in ARCHIVOS.items():
    destino = EJECUCION / nombre
    if not destino.exists() or huella(destino) != huellas[nombre]:
        temporal = destino.with_suffix(destino.suffix + ".tmp")
        shutil.copy2(origen, temporal)
        temporal.replace(destino)
country = EJECUCION / "country.csv"
if not country.exists():
    url = f"https://raw.githubusercontent.com/ladmepaz/bibfusion/{REVISION}/data/country.csv"
    with urllib.request.urlopen(url, timeout=60) as r:
        datos = r.read()
    tmp = country.with_suffix(".tmp")
    tmp.write_bytes(datos)
    tmp.replace(country)
if not {"Name", "Alpha-2", "Alpha-3"}.issubset(pd.read_csv(country, sep=";", nrows=0).columns):
    raise ValueError("country.csv no tiene el formato esperado.")
manifest = dict(identidad, python=platform.python_version(),
                dependencias={p: version(p) for p in ["bibfusion", "pandas", "numpy", "requests", "rapidfuzz"]},
                preparado_utc=datetime.now(timezone.utc).isoformat(),
                scopus_registros=len(scopus), wos_registros=sum(s.strip() == "ER" for s in lineas),
                country_sha256=huella(country))
(EJECUCION / "configuracion.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
(EJECUCION / "requirements.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
print("Scopus:", len(scopus), "registros; WoS:", manifest["wos_registros"], "registros.")
print("Preparación terminada. Resultados en:", EJECUCION)
del scopus, wos_texto, lineas


## 4. Ejecutar
La clave se pide en un campo oculto. Esta edición requiere clave para evitar consultas anónimas accidentales.
El registro muestra inicio y fin de cada etapa de consulta, sin prometer porcentaje ni tiempo restante.
Los errores de la biblioteca se conservan en `registro_….txt` con la clave ocultada.

Si se interrumpe, repite 1–4 con las mismas entradas. No necesitas borrar archivos ni cambiar nombres.
Para repetir consultas con nuevas respuestas, desmarca **REUTILIZAR_ETAPAS**. Esto repetirá las etapas y actualizará sus resultados.


In [ ]:
#@title 4a. Preparar registro y recuperación (ejecutar antes de 4b)
import contextlib, functools, gzip, importlib, inspect, io, os, pickle, re, time
from urllib.parse import quote

class RegistroSeguro(io.TextIOBase):
    """Oculta secretos incluso cuando print los escribe en fragmentos."""
    def __init__(self, pantalla, archivo, secreto):
        self.pantalla, self.archivo = pantalla, archivo
        self.secretos = [s for s in (secreto, quote(secreto, safe="")) if s]
        self.pendiente = ""
    def limpiar(self, texto):
        for secreto in self.secretos:
            texto = texto.replace(secreto, "[CLAVE_OCULTA]")
        return re.sub(r"(?i)(api_key=)[^\s&]+", r"\1[CLAVE_OCULTA]", texto)
    def write(self, texto):
        self.pendiente += str(texto)
        while "\n" in self.pendiente:
            linea, self.pendiente = self.pendiente.split("\n", 1)
            self.emitir(linea + "\n")
        return len(texto)
    def emitir(self, texto):
        texto = self.limpiar(texto)
        self.pantalla.write(texto)
        self.archivo.write(texto)
        self.pantalla.flush()
        self.archivo.flush()
    def flush(self):
        # No mostrar fragmentos: podrían contener parte de una clave.
        if not self.archivo.closed:
            self.archivo.flush()
    def terminar(self):
        if self.pendiente:
            self.emitir(self.pendiente + "\n")
            self.pendiente = ""

def corregir_nulos_wos():
    modulo = importlib.import_module("bibfusion.modules.enrich_references_with_openalex")
    if getattr(modulo, "_parche_docente", False):
        return
    fuente = inspect.getsource(modulo.get_paper_info_from_doi)
    cambios = {
        'data.get("primary_location", {}).get("source", {})': '(data.get("primary_location") or {}).get("source") or {}',
        'data.get("biblio", {})': '(data.get("biblio") or {})',
        'data.get("authorships", [])': '(data.get("authorships") or [])',
        'auth.get("author", {})': '(auth.get("author") or {})',
        'author_data.get("display_name", "")': '(author_data.get("display_name") or "")',
        'author_data.get("id", "")': '(author_data.get("id") or "")',
        'auth.get("institutions", [])': '(auth.get("institutions") or [])',
        'data.get("keywords", [])': '(data.get("keywords") or [])',
    }
    for anterior, nuevo in cambios.items():
        if fuente.count(anterior) != 1:
            raise RuntimeError("La versión instalada no coincide con el parche docente. Reinicia la sesión e instala la revisión indicada.")
        fuente = fuente.replace(anterior, nuevo)
    exec(compile(fuente, "<parche_docente_nulos_wos>", "exec"), modulo.__dict__)
    modulo._parche_docente = True

ETAPAS = ["enrich_wos_with_openalex_authors", "enrich_references_with_openalex",
          "enrich_scopus_with_openalex_authors", "generate_references_column", "openalex_enrich_ref"]

def con_recuperacion(funcion, carpeta, reutilizar):
    @functools.wraps(funcion)
    def envolver(*args, **kwargs):
        inicio = time.monotonic()
        parametros = inspect.signature(funcion).bind(*args, **kwargs)
        # La clave nunca forma parte de los archivos guardados.
        entradas = {k: v for k, v in parametros.arguments.items() if "key" not in k.lower()}
        digest = hashlib.sha256(pickle.dumps(entradas, protocol=5)).hexdigest()
        destino = carpeta / (funcion.__name__ + "_" + digest + ".pkl.gz")
        if reutilizar and destino.exists():
            try:
                with gzip.open(destino, "rb") as f:
                    resultado = pickle.load(f)
                if resultado is not None:
                    print("RECUPERADA:", funcion.__name__)
                    return resultado
            except (EOFError, OSError, pickle.UnpicklingError, ImportError, ValueError):
                print("Punto de recuperación ilegible; se repetirá:", funcion.__name__)
        print("INICIO:", funcion.__name__, "— puede tardar; espera al mensaje FIN.")
        resultado = funcion(*args, **kwargs)
        if resultado is None:
            raise RuntimeError("La etapa " + funcion.__name__ + " no devolvió datos.")
        temporal = destino.with_suffix(destino.suffix + ".tmp")
        with gzip.open(temporal, "wb") as f:
            pickle.dump(resultado, f, protocol=5)
        temporal.replace(destino)
        print("FIN:", funcion.__name__, "—", round((time.monotonic()-inicio)/60, 1), "minutos; recuperación guardada.")
        return resultado
    return envolver

ESPERADOS = ["All_Articles.csv", "All_Authors.csv", "All_ArticleAuthor.csv", "All_Citation.csv",
             "All_Affiliation.csv", "All_Journal.csv", "All_Scimagodb.csv"]

def validar_salidas(carpeta):
    filas = []
    for nombre in ESPERADOS:
        archivo = carpeta / nombre
        if not archivo.is_file() or archivo.stat().st_size == 0:
            raise RuntimeError("Integración incompleta: falta " + nombre)
        total = 0
        columnas = list(pd.read_csv(archivo, nrows=0).columns)
        for bloque in pd.read_csv(archivo, chunksize=20000, dtype=str):
            total += len(bloque)
        filas.append({"Archivo": nombre, "Filas": total, "Columnas": len(columnas)})
    vacias = [r["Archivo"] for r in filas if r["Filas"] == 0]
    if vacias:
        raise RuntimeError("Se generaron tablas vacías; revisa el registro: " + str(vacias))
    return pd.DataFrame(filas)

print("Herramientas preparadas. Ejecuta 4b.")


In [ ]:
#@title 4b. Ejecutar BibFusion y guardar resultados
from getpass import getpass
import requests

def ejecutar():
    resultados = EJECUCION / "all_data_wos_scopus"
    terminado = EJECUCION / "COMPLETADO.json"
    if terminado.exists() and REUTILIZAR_ETAPAS:
        validar_salidas(resultados)
        print("Esta entrada ya tiene resultados completos. Continúa con la celda 5.")
        return
    clave = getpass("Pega tu clave de OpenAlex y presiona Enter: ").strip()
    if not clave:
        raise ValueError("Se necesita tu clave de OpenAlex para esta edición del cuaderno.")
    # Verificar acceso antes de iniciar un trabajo de varias horas.
    try:
        r = requests.get("https://api.openalex.org/works", params={"per-page": 1, "api_key": clave}, timeout=30)
        estado = r.status_code
        r.close()
    except requests.RequestException:
        raise RuntimeError("No se pudo conectar con OpenAlex. Revisa la conexión y vuelve a intentar.") from None
    if estado != 200:
        raise RuntimeError(f"OpenAlex respondió HTTP {estado}. Revisa tu clave o cuota antes de repetir.")
    del r
    corregir_nulos_wos()
    pipeline = importlib.import_module("bibfusion.pipeline")
    # Separar caches si cambia pandas: los pickles no son un formato de intercambio.
    carpeta_cache = EJECUCION / "recuperacion" / ("pandas_" + pd.__version__)
    carpeta_cache.mkdir(parents=True, exist_ok=True)
    originales = {nombre: getattr(pipeline, nombre) for nombre in ETAPAS}
    log = EJECUCION / ("registro_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f") + ".txt")
    fallo = None
    if terminado.exists():
        terminado.unlink()
    with open(log, "w", encoding="utf-8") as archivo:
        seguro = RegistroSeguro(sys.stdout, archivo, clave)
        try:
            with contextlib.redirect_stdout(seguro), contextlib.redirect_stderr(seguro):
                print("Versión docente:", VERSION_DOCENTE, "Revisión:", REVISION)
                print("Parche: campos nulos en referencias WoS; vacíos conservados como vacíos.")
                for nombre, funcion in originales.items():
                    setattr(pipeline, nombre, con_recuperacion(funcion, carpeta_cache, REUTILIZAR_ETAPAS))
                pipeline.preprocessing_df(
                    path_wos=str(EJECUCION / "wos.txt"),
                    path_scopus=str(EJECUCION / "scopus.csv"),
                    path_scimago=str(EJECUCION / "scimago.csv"),
                    path_country=str(EJECUCION / "country.csv"),
                    API_KEY_OPENALEX=clave,
                )
                resumen = validar_salidas(resultados)
                resumen.to_csv(EJECUCION / "resumen_tablas.csv", index=False)
                terminado.write_text(json.dumps({"fin_utc": datetime.now(timezone.utc).isoformat(),
                    "estado": "archivos_generados_pendientes_de_auditoria"}, indent=2), encoding="utf-8")
                print("TERMINADO: siete tablas generadas y legibles. Falta auditar su calidad.")
        except (Exception, KeyboardInterrupt) as e:
            fallo = seguro.limpiar(type(e).__name__ + ": " + str(e))
            seguro.write("\nEJECUCIÓN INTERRUMPIDA: " + fallo + "\n")
        finally:
            for nombre, funcion in originales.items():
                setattr(pipeline, nombre, funcion)
            seguro.terminar()
            seguro.secretos.clear()
            clave = ""
    print("Registro:", log)
    if fallo:
        raise RuntimeError(fallo + " — Corrige la causa y repite las celdas; las etapas completas se conservan.") from None
    print("Resultados:", resultados)

ejecutar()


In [ ]:
#@title 5. Ver el resumen y localizar los resultados
from IPython.display import display
RESULTADOS = EJECUCION / "all_data_wos_scopus"
if not (EJECUCION / "COMPLETADO.json").exists():
    raise RuntimeError("Esta ejecución no terminó correctamente. Revisa el registro antes de usar las tablas.")
resumen = validar_salidas(RESULTADOS)
display(resumen)
print("Carpeta:", RESULTADOS)
print("Los originales y resultados permanecen en Drive al cerrar Colab.")
print("All_Articles incluye registros principales y referencias: sus filas NO son el tamaño del corpus principal.")
print("Antes del análisis revisa duplicados, identificación de autores, años y errores del registro.")


## Si aparece un problema
| Mensaje o situación | Qué hacer |
|---|---|
| Drive already mounted | Es informativo; continúa. |
| No encuentra los archivos | Revisa la cuenta montada, CARPETA y los nombres de archivo de la celda 2. |
| Formato SCImago incorrecto | Solicita el auxiliar preparado al docente. No inventes columnas ni uses un archivo vacío. |
| 401 / 403 / 429 | Revisa clave, permisos o cuota de OpenAlex. No relances en bucle. |
| 404 durante el procesamiento | Algunos registros no se encontraron. Debes evaluar su cobertura al terminar. |
| Colab se desconectó | Ejecuta nuevamente 1–4 con la misma configuración. La etapa incompleta se repetirá. |
| Se detuvo con otro error | Conserva registro e intermedios y compártelos con el docente, sin claves. |
| Las tablas existen pero están vacías | No las uses como resultados válidos. Revisa el registro. |

## Para el docente
- Prueba primero el cuaderno con exportaciones pequeñas y el auxiliar que distribuirás. La preparación y los mecanismos de recuperación/protección se verificaron con pruebas locales; **no se ejecutó aquí un procesamiento completo de varias horas en Colab**.
- Corrección aplicada: `primary_location`, `source`, `biblio`, `author`, listas e identificadores nulos en `get_paper_info_from_doi`. Los nombres/IDs ausentes quedan vacíos conservando sus posiciones; no se inventa información. Otros defectos de la biblioteca pueden persistir.
- Los puntos de recuperación son retornos de cinco funciones; pueden incluir enriquecimiento parcial si BibFusion manejó internamente un error. Desmarca REUTILIZAR_ETAPAS para renovarlos después de corregir la causa. Los registros de cada intento se conservan.
- La recuperación verifica entradas por contenido y revisión, pero **no es un checkpoint de cada solicitud**. No elimina el tiempo de espera ni garantiza continuidad en Colab.
- No distribuyas claves ni los archivos de recuperación `.pkl.gz`. Comparte el cuaderno sin salidas personales. El cuaderno no modifica el repositorio oficial.
- Para reproducir un estudio, conserva exportaciones originales, fecha y consulta de búsqueda, auxiliar SCImago, configuración, requirements, registros y tablas finales. Documenta esta corrección docente, pues difiere de la revisión original.

Fuentes: [BibFusion, revisión utilizada](https://github.com/ladmepaz/bibfusion/tree/ffda6561f812a2645c9fb234fa0536a798d694e3) · [Condiciones de Colab](https://research.google.com/colaboratory/faq.html) · [Clave OpenAlex](https://openalex.org/settings/api).


# Parte 2 · Corpus principal y equipos de autores
**Qué hace esta parte:** (1) lee las tablas que produjo BibFusion; (2) detecta artículos duplicados entre WoS y Scopus, con coincidencia aproximada de títulos y revisión manual; (3) aplica las equivalencias confirmadas a mano; (4) asigna a cada autor un identificador único (ORCID, luego OpenAlex, luego nombre normalizado) y une los identificadores de una misma persona; (5) guarda `corpus_principal_442.csv` y `equipos_personas_por_articulo.csv` en `trabajo_limpio/`.

**Antes de ejecutar:** la ruta de la primera celda incluye el nombre de la carpeta de ejecución de la Parte 1 (`ejecucion_…`). Si la tuya tiene otro nombre, cámbialo.


In [ ]:
import pandas as pd
from pathlib import Path
R = Path("/content/drive/MyDrive/BibFusion/salidas/ejecucion_a8dc08ef4f119c7b/all_data_wos_scopus")
A  = pd.read_csv(R/"All_Articles.csv", dtype=str, keep_default_na=False)
AA = pd.read_csv(R/"All_ArticleAuthor.csv", dtype=str, keep_default_na=False)
Au = pd.read_csv(R/"All_Authors.csv", dtype=str, keep_default_na=False)
Af = pd.read_csv(R/"All_Affiliation.csv", dtype=str, keep_default_na=False)
M = A[A["__is_main"] == "True"]
print("Principales:", len(M), "| SR únicos:", M["SR"].nunique())
print(M[["ismain_wos","ismain_scopus"]].value_counts())
print("Años:", M["year"].astype(float).agg(["min","max"]).tolist())
print(M["document_type"].value_counts())
d = M[M["__doi_norm"]!=""]; print("DOI repetido entre principales:", d["__doi_norm"].duplicated().sum())
print("PersonID por tipo:", AA["PersonID"].str.extract(r"^([A-Za-z]+):")[0].value_counts().to_dict())
print("Autor de correspondencia:", AA["CorrespondingAuthor"].value_counts().to_dict())
print("Afiliación vacía:", (Af["Affiliation"]=="NO AFFILIATION").mean().round(3), "| país vacío:", (Af["Country"]=="NO COUNTRY").mean().round(3))

In [ ]:
from itertools import combinations
M = A[A["__is_main"]=="True"].copy(); M["year"] = M["year"].astype(float).astype(int)
print(M[M["SR"].duplicated(keep=False)][["SR","__doi_norm","year","journal","sources_merged"]].to_string())
print("Principales sin DOI:", (M["__doi_norm"]=="").sum())
print(M["year"].value_counts().sort_index().to_string())
P = AA[AA["SR"].isin(M["SR"])]
print("Autores-artículo:", len(P), "| personas únicas:", P["PersonID"].nunique())
print(P["PersonID"].str.extract(r"^([A-Za-z]+):")[0].value_counts(normalize=True).round(3).to_dict())
Pf = Af[Af["SR"].isin(M["SR"])]
print("Afiliación vacía:", (Pf["Affiliation"]=="NO AFFILIATION").mean().round(3), "| país vacío:", (Pf["Country"]=="NO COUNTRY").mean().round(3))
yr = M.drop_duplicates("SR").set_index("SR")["year"]
eq = P.groupby("SR")["PersonID"].apply(lambda s: sorted(set(s)))
pares = {}
for sr, ids in eq.items():
    for a, b in combinations(ids, 2): pares.setdefault((a, b), []).append(yr[sr])
tot = hist = 0
for sr, ids in eq.items():
    if len(ids) < 2: continue
    tot += 1; t = yr[sr]
    if any(any(t-5 <= y < t for y in pares[(a, b)]) for a, b in combinations(ids, 2)): hist += 1
print("Equipos con 2+ autores:", tot, "| con algún par con historia previa (5 años):", hist)


In [ ]:
!pip install -q rapidfuzz


In [ ]:
import pandas as pd
from itertools import combinations
from rapidfuzz import fuzz
print(M[M["SR"].duplicated(keep=False)][["SR","title","author","doi"]].to_string())
W = M[(M["ismain_wos"]=="True") & (M["ismain_scopus"]!="True")]
S = M[(M["ismain_scopus"]=="True") & (M["ismain_wos"]!="True")]
res = []
for _, w in W.iterrows():
    for _, s in S.iterrows():
        if abs(w["year"]-s["year"]) <= 1:
            sc = fuzz.token_set_ratio(str(w["title"]), str(s["title"]))
            if sc >= 90: res.append((sc, w["SR"], s["SR"], str(w["title"])[:60]))
for r in sorted(res, reverse=True): print(r)
A["year_i"] = pd.to_numeric(A["year"], errors="coerce")
yr_all = A.drop_duplicates("SR").set_index("SR")["year_i"]
eq_all = AA.groupby("SR")["PersonID"].apply(lambda s: sorted(set(s)))
pares = {}
for sr, ids in eq_all.items():
    y = yr_all.get(sr)
    if pd.isna(y): continue
    for a, b in combinations(ids, 2): pares.setdefault((a, b), []).append(y)
tot = hist = 0
for sr in M["SR"].drop_duplicates():
    ids = eq_all.get(sr, [])
    if len(ids) < 2: continue
    t = yr[sr]; tot += 1
    if any(any(t-5 <= y < t for y in pares.get((a, b), [])) for a, b in combinations(ids, 2)): hist += 1
print("Con corpus + referencias -> equipos 2+:", tot, "| con historia previa:", hist)

In [ ]:
import pandas as pd

# WoS -> Scopus (se conserva el SR de Scopus). Solo pares confirmados a mano.
equivalencias = {
 "SCBINDEBUTTE M, 2008, J SMALL BUS MANAGE": "SCHINDEHUTTE M, 2008, J SMALL BUS MANAGE",
 "OLIVA-BECERRA I, 2012, REV CIENC SOC-VENEZ": "OLIVA-BECERRA I, 2012, REV CIENC SOC",
 "NWANKWO CA, 2020, INT J ENTREP VENTUR": "NWANKWO C, 2020, INT J ENTREP VENTURING",
 "MORARU AD, 2017, TRANSFORM BUS ECON": "MORARU A, 2017, TRANSFORM BUS ECON",
 "MONFERRER D, 2012, MARK-TRZ": "MONFERRER D, 2012",
 "MIRZAEI O, 2016, INT FOOD AGRIBUS MAN": "MIRZAEI O, 2016",
 "MASSIERA P, 2024, REV INT PME": "MASSIERA P, 2024, RNA",
 "IBRAHIM MA, 2016, INT J BUS SOC": "IBRAHIM M, 2016",
 "DJASTUTI I, 2020, QUAL-ACCESS SUCCESS": "DJASTUTI I, 2020, QUAL ACCESS SUCCESS",
 "ALEXIOU K, 2022, INT J ARTS MANAG": "ALEXIOU K, 2022",
 "ADAWIAH R, 2022, EURASIAN J EDUC RES": "ADAWIAH R, 2022",
 "GARCIA-RODRIGUEZ FJ, 2014, INTERCIENCIA": "GARCIA-RODRIGUEZ F, 2014, INTERCIENCIA",
 "SLOGAR H, 2020, EKON MISAO PRAKSA": "SLOGAR H, 2020",
}
mismo_sr = ["FOLTEAN F, 2009, TRANSFORM BUS ECON"]  # WoS y Scopus con el mismo SR

n0 = len(M)
Mc = M[~M["SR"].isin(equivalencias)].copy()
# AYDI: conservar la versión final (DOI inno.055.0199)
quitar = (Mc["SR"] == "AYDI I, 2018, INNOVATIONS") & (Mc["__doi_norm"].str.lower() != "10.3917/inno.055.0199")
Mc = Mc[~quitar].drop_duplicates("SR")
# procedencia: los fusionados quedan en ambas bases
fusionados = list(equivalencias.values()) + mismo_sr
Mc.loc[Mc["SR"].isin(fusionados), ["ismain_wos", "ismain_scopus"]] = "True"

AAc = AA.assign(SR=AA["SR"].replace(equivalencias)).drop_duplicates(["SR", "PersonID"])
Afc = Af.assign(SR=Af["SR"].replace(equivalencias)).drop_duplicates(["SR", "AuthorID"])

print("Principales antes:", n0, "| después:", len(Mc), "| SR únicos:", Mc["SR"].nunique(), "| SR repetidos:", Mc["SR"].duplicated().sum())
print(Mc[["ismain_wos", "ismain_scopus"]].value_counts())
chk = AAc[AAc["SR"].isin(fusionados)].groupby("SR")["PersonID"].nunique()
print("Autores por artículo fusionado:\n", chk.to_string())

In [ ]:
base = AA[~AA["SR"].isin(equivalencias)]               # quita la versión WoS de los 13 pares
esFus = base["SR"].isin(fusionados)
AAc = pd.concat([base[~esFus], base[esFus].drop_duplicates(["SR", "AuthorOrder"])])
Afc = Af[~Af["SR"].isin(equivalencias)]

# Auditoría de identidad en los principales
P = AAc[AAc["SR"].isin(Mc["SR"])]
rep = P[P.duplicated(["SR", "PersonID"], keep=False)].sort_values(["SR", "AuthorOrder"])
print("Artículos con la misma persona en 2+ posiciones:", rep["SR"].nunique())
print(rep[["SR", "AuthorOrder", "PersonID"]].head(40).to_string())

n_orig = Mc.set_index("SR")["author"].str.split(";").str.len()
n_aa = P.groupby("SR").size()
dif = pd.concat([n_orig.rename("autores_en_registro"), n_aa.rename("filas_en_ArticleAuthor")], axis=1)
dif = dif[dif["autores_en_registro"] != dif["filas_en_ArticleAuthor"]]
print("Artículos donde no coincide el número de autores:", len(dif))
print(dif.head(30).to_string())

Au["apellido"] = Au["AuthorFullName"].str.upper().str.replace(r"[^A-Z ]", "", regex=True).str.split().str[0]
multi = Au.groupby("PersonID")["apellido"].nunique()
print("PersonID con 2+ apellidos distintos:", (multi > 1).sum(), "de", len(multi))
print(Au[Au["PersonID"].isin(multi[multi > 1].index)].sort_values("PersonID")[["PersonID", "AuthorFullName"]].head(30).to_string())


In [ ]:
from itertools import combinations
P = AAc[AAc["SR"].isin(Mc["SR"])].copy()
P["ord"] = pd.to_numeric(P["AuthorOrder"], errors="coerce")
T = P.groupby(["SR", "PersonID"], as_index=False)["ord"].min().sort_values(["SR", "ord"])
T["pos"] = T.groupby("SR").cumcount() + 1
n_orig = Mc.set_index("SR")["author"].str.split(";").str.len()
chk = T.groupby("SR").size().rename("personas").to_frame().join(n_orig.rename("registro"))
chk["dif"] = chk["personas"] - chk["registro"]
print(chk["dif"].value_counts().sort_index().to_string())
print(chk[chk["dif"] != 0].sort_values("dif").head(25).to_string())

def diag(Mx, AAx, solo_corpus=True):
    yrx = A.assign(y=pd.to_numeric(A["year"], errors="coerce")).drop_duplicates("SR").set_index("SR")["y"]
    base = AAx[AAx["SR"].isin(Mx["SR"])] if solo_corpus else AAx
    eq = base.groupby("SR")["PersonID"].apply(lambda s: sorted(set(s)))
    pares = {}
    for sr, ids in eq.items():
        y = yrx.get(sr)
        if pd.isna(y): continue
        for a, b in combinations(ids, 2): pares.setdefault((a, b), []).append(y)
    yf = Mx.set_index("SR")["year"].astype(float)
    tot = hist = 0
    for sr in Mx["SR"]:
        ids = eq.get(sr, [])
        if len(ids) < 2: continue
        t = yf[sr]; tot += 1
        if any(any(t-5 <= y < t for y in pares.get((a, b), [])) for a, b in combinations(ids, 2)): hist += 1
    return tot, hist

AAfinal = pd.concat([AAc[~AAc["SR"].isin(Mc["SR"])][["SR", "PersonID"]], T[["SR", "PersonID"]]])
print("Solo corpus:", diag(Mc, AAfinal, True))
print("Corpus + referencias:", diag(Mc, AAfinal, False))

In [ ]:
import unicodedata, re
prio = {"ORCID": 0, "OA": 1, "NAME": 2, "NAMEEMAIL": 3}
rango = lambda p: prio.get(p.split(":")[0], 9)
def toks(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper()
    return {t for t in re.sub(r"[^A-Z ]", " ", s).split() if len(t) >= 2}
nombres = Au.groupby("PersonID")["AuthorFullName"].apply(lambda s: set().union(*[toks(x) for x in s]))

padre = {}
def raiz(x):
    padre.setdefault(x, x)
    while padre[x] != x:
        padre[x] = padre[padre[x]]; x = padre[x]
    return x
def unir(a, b):
    ra, rb = raiz(a), raiz(b)
    if ra == rb: return
    if (rango(ra), ra) <= (rango(rb), rb): padre[rb] = ra
    else: padre[ra] = rb
def mismo(a, b):
    X, Y = nombres.get(a, set()), nombres.get(b, set())
    inter = X & Y
    if len(inter) >= 2: return True
    return len(inter) == 1 and min(len(X), len(Y)) == 1 and len(next(iter(inter))) >= 4

for sr, g in T.groupby("SR"):
    for a, b in combinations(g["PersonID"].tolist(), 2):
        if mismo(a, b): unir(a, b)

mapa = lambda x: raiz(x) if x in padre else x
T2 = T.assign(PersonID=T["PersonID"].map(mapa)).groupby(["SR", "PersonID"], as_index=False)["ord"].min().sort_values(["SR", "ord"])
T2["pos"] = T2.groupby("SR").cumcount() + 1
chk2 = T2.groupby("SR").size().rename("personas").to_frame().join(n_orig.rename("registro"))
chk2["dif"] = chk2["personas"] - chk2["registro"]
print(chk2["dif"].value_counts().sort_index().to_string())
print(chk2[chk2["dif"] != 0].sort_values("dif").head(25).to_string())

refs = AAc[~AAc["SR"].isin(Mc["SR"])][["SR", "PersonID"]].assign(PersonID=lambda d: d["PersonID"].map(mapa))
AAfinal2 = pd.concat([refs, T2[["SR", "PersonID"]]])
print("Solo corpus:", diag(Mc, AAfinal2, True))
print("Corpus + referencias:", diag(Mc, AAfinal2, False))

In [ ]:
padre.clear()
for sr, g in T.groupby("SR"):
    ids = g["PersonID"].tolist()
    n = n_orig.get(sr)
    if pd.isna(n): continue
    exceso = len(ids) - int(n)
    if exceso <= 0: continue
    cand = sorted(((len(nombres.get(a, set()) & nombres.get(b, set())), a, b)
                   for a, b in combinations(ids, 2) if mismo(a, b)), reverse=True)
    for _, a, b in cand[:exceso]: unir(a, b)

mapa = lambda x: raiz(x) if x in padre else x
T2 = T.assign(PersonID=T["PersonID"].map(mapa)).groupby(["SR", "PersonID"], as_index=False)["ord"].min().sort_values(["SR", "ord"])
T2["pos"] = T2.groupby("SR").cumcount() + 1
chk2 = T2.groupby("SR").size().rename("personas").to_frame().join(n_orig.rename("registro"))
chk2["dif"] = chk2["personas"] - chk2["registro"]
print(chk2["dif"].value_counts().sort_index().to_string())
refs = AAc[~AAc["SR"].isin(Mc["SR"])][["SR", "PersonID"]].assign(PersonID=lambda d: d["PersonID"].map(mapa))
AAfinal2 = pd.concat([refs, T2[["SR", "PersonID"]]])
print("Solo corpus:", diag(Mc, AAfinal2, True))
print("Corpus + referencias:", diag(Mc, AAfinal2, False))

In [ ]:
from pathlib import Path
OUT = Path("/content/drive/MyDrive/BibFusion/trabajo_limpio"); OUT.mkdir(parents=True, exist_ok=True)
Mc.to_csv(OUT/"corpus_principal_442.csv", index=False)
T2.to_csv(OUT/"equipos_personas_por_articulo.csv", index=False)
chk2[chk2["dif"] != 0].sort_values("dif").to_csv(OUT/"revision_manual_equipos.csv")
pd.Series(equivalencias, name="SR_canonico").rename_axis("SR_reemplazado").to_csv(OUT/"equivalencias_duplicados.csv")
pd.Series({k: raiz(k) for k in list(padre) if raiz(k) != k}, name="PersonID_canonico").rename_axis("PersonID_original").to_csv(OUT/"equivalencias_personas.csv")
print(sorted(p.name for p in OUT.iterdir()))

# Parte 3 · Referencias citadas
**Qué hace esta parte:** toma las referencias de cada artículo del corpus, elimina duplicados, normaliza autores, año y revista de cada referencia y guarda `citas_principales_final.csv`.


In [ ]:
import pandas as pd
from pathlib import Path
R = Path("/content/drive/MyDrive/BibFusion/salidas/ejecucion_a8dc08ef4f119c7b/all_data_wos_scopus")
Ci = pd.read_csv(R/"All_Citation.csv", dtype=str, keep_default_na=False)
Ci = Ci[Ci["SR"].isin(Mc["SR"])].copy()
info = A.drop_duplicates("SR").set_index("SR")[["year", "source_title", "journal"]]
Ci = Ci.join(info, on="SR_ref")
ok = lambda s: s.notna() & s.ne("")
Ci["anio_ok"] = ok(Ci["year"])
Ci["rev_ok"] = ok(Ci["source_title"]) | ok(Ci["journal"])
Ci = Ci.join(Mc.set_index("SR")[["year", "ismain_wos", "ismain_scopus"]].rename(columns={"year": "anio_art"}), on="SR")
Ci["base"] = "solo Scopus"
Ci.loc[(Ci["ismain_wos"] == "True") & (Ci["ismain_scopus"] == "True"), "base"] = "ambas"
Ci.loc[(Ci["ismain_wos"] == "True") & (Ci["ismain_scopus"] != "True"), "base"] = "solo WoS"

por_art = Ci.groupby("SR").size()
print("Principales sin ninguna referencia en Citation:", len(Mc) - len(por_art))
print(por_art.describe().round(1).to_string())
print("\nCobertura por procedencia:")
print(Ci.groupby("base")[["anio_ok", "rev_ok"]].agg(["mean", "size"]).round(3).to_string())
Ci["periodo"] = pd.cut(pd.to_numeric(Ci["anio_art"], errors="coerce"), [1999, 2009, 2014, 2019, 2022, 2026],
                       labels=["2000-09", "2010-14", "2015-19", "2020-22", "2023-26"])
print("\nCobertura por periodo del artículo:")
print(Ci.groupby("periodo", observed=True)[["anio_ok", "rev_ok"]].agg(["mean", "size"]).round(3).to_string())
cob = Ci.groupby("SR")["rev_ok"].mean()
print("\nArtículos con menos de 50% de referencias con revista:", int((cob < 0.5).sum()), "de", len(cob))
print("Artículos con menos de 80%:", int((cob < 0.8).sum()))

In [ ]:
Cd = Ci.drop_duplicates(["SR", "SR_ref"]).copy()
print("Filas:", len(Ci), "-> sin duplicados:", len(Cd))
print(Cd.groupby("SR").size().describe().round(1).to_string())
print(Cd.groupby("base")[["anio_ok", "rev_ok"]].agg(["mean", "size"]).round(3).to_string())
Cd["anio_txt"] = Cd["SR_ref"].str.extract(r",\s*((?:19|20)\d{2})\b")[0].notna()
print("Referencias con año legible en el texto:", Cd["anio_txt"].mean().round(3))

sin = Mc[~Mc["SR"].isin(por_art.index)]
print("\nSin referencias, por procedencia:\n", sin[["ismain_wos", "ismain_scopus"]].value_counts().to_string())
print(pd.to_numeric(sin["year"], errors="coerce").value_counts().sort_index().to_string())
print("Con texto en 'references':", int((sin["references"].str.len() > 0).sum()),
      "| con cited_reference_count > 0:", int((pd.to_numeric(sin["cited_reference_count"], errors="coerce") > 0).sum()))

In [ ]:
import re, unicodedata
from difflib import SequenceMatcher
PART = {"DE","DEL","DA","DO","DOS","VAN","VON","AL","EL","DI","LA","LE","DER","DEN","BIN","BEN"}
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().upper()
    return re.sub(r"[^A-Z0-9 ,]", " ", s)
def k_autor(sr_ref):
    aut = norm(sr_ref).split(",")[0].split()
    if len(aut) > 1: aut = aut[:-1]                # quita las iniciales
    return " ".join([t for t in aut if t not in PART] or aut)
def k_fuente(sr_ref):
    p = norm(sr_ref).split(",")
    return " ".join(p[2:]).strip() if len(p) > 2 else ""
Cd["k_aut"] = Cd["SR_ref"].map(k_autor)
Cd["k_fuente"] = Cd["SR_ref"].map(k_fuente)
Cd["k_anio"] = pd.to_numeric(Cd["SR_ref"].str.extract(r",\s*((?:19|20)\d{2})\b")[0], errors="coerce")
def iguales(a, b):
    if pd.notna(a["k_anio"]) and pd.notna(b["k_anio"]) and abs(a["k_anio"] - b["k_anio"]) > 1: return False
    if a["k_fuente"] and b["k_fuente"]:
        return SequenceMatcher(None, a["k_fuente"], b["k_fuente"]).ratio() >= 0.6
    return True
Cd = Cd.sort_values(["rev_ok", "anio_ok"], ascending=False, kind="stable")
marca = pd.Series(False, index=Cd.index)
for (sr, ka), g in Cd.groupby(["SR", "k_aut"]):
    if len(g) < 2: continue
    kept = []
    for idx, r in g.iterrows():
        if any(iguales(r, k) for k in kept): marca[idx] = True
        else: kept.append(r)
Cu = Cd[~marca]
print("Filas:", len(Cd), "->", len(Cu))
print(Cu.groupby(["SR", "base"]).size().reset_index(name="n").groupby("base")["n"].describe().round(1).to_string())
print(Cu.groupby("base")[["anio_ok", "rev_ok"]].mean().round(3).to_string())
print(Cd[marca].sample(15, random_state=1)[["SR", "SR_ref"]].to_string())

In [ ]:
import re
import pandas as pd

SPLIT = re.compile(r"(?<=\((?:19|20)\d\d\));\s+")
AUT = re.compile(r"^[^\d,;()]+\s[A-ZÀ-Þ]\.(?:[A-ZÀ-Þ]\.)*$")      # APELLIDO I.I.

def base_aut(aut):
    t = aut.replace(".", " ").split()
    i = next((k for k, x in enumerate(t) if len(x) == 1), len(t))
    return f"{' '.join(t[:i])} {''.join(t[i:])}".strip()

def parse_item(item):
    item = item.strip()
    m = re.search(r"\(((?:19|20)\d\d)\)\s*$", item)
    anio = m.group(1) if m else ""
    cuerpo = item[:m.start()].rstrip(", ").strip() if m else item
    head, _, resto = cuerpo.partition(", ")
    primero = head.split("; ")[0].strip()
    if AUT.match(primero):
        aut = primero
    else:
        aut, resto = "", cuerpo
    partes = [p.strip() for p in resto.split(", ")] if resto else []
    fuente = ""
    for i, p in enumerate(partes):
        if i >= 1 and re.fullmatch(r"\d+", p):
            fuente = partes[i-1]; break
    else:
        for i, p in enumerate(partes):
            if i >= 1 and p.startswith("PP."):
                fuente = partes[i-1]; break
    base = base_aut(aut) if aut else "ANONYMOUS"
    sr_ref = base + (f", {anio}" if anio else "") + (f", {fuente}" if fuente else "")
    return aut, anio, fuente, sr_ref

# 1) Recuperar las referencias de los artículos sin citas
sin = Mc[~Mc["SR"].isin(Cu["SR"])].copy()
filas = []
for _, r in sin.iterrows():
    for it in SPLIT.split(str(r["references"])):
        if it.strip():
            aut, anio, fuente, sr_ref = parse_item(it)
            filas.append((r["SR"], sr_ref, anio, fuente, it.strip()[:130]))
Rec = pd.DataFrame(filas, columns=["SR", "SR_ref", "anio", "fuente", "item"])

# 2) Validación
pa = Rec.groupby("SR").size()
print("Referencias recuperadas:", len(Rec), "| artículos:", pa.shape[0], "de", len(sin))
print(pa.describe().round(1).to_string())
print("Con año:", round((Rec["anio"] != "").mean(), 3), "| con revista:", round((Rec["fuente"] != "").mean(), 3),
      "| sin autor:", round(Rec["SR_ref"].str.startswith("ANONYMOUS").mean(), 3))
print(Rec[Rec["item"].str.startswith("IKA")][["SR_ref", "item"]].to_string())

# 3) Unir a la tabla de citas y guardar
Cu["origen"] = "citas_BibFusion"
Rec2 = Rec[["SR", "SR_ref"]].assign(origen="texto_references", anio_ok=True, rev_ok=Rec["fuente"].ne(""))
Cfinal = pd.concat([Cu[["SR", "SR_ref", "anio_ok", "rev_ok", "origen"]], Rec2], ignore_index=True)
print(Cfinal["SR"].nunique(), "artículos con referencias, de", len(Mc))
print(Cfinal.groupby("origen").size().to_string())
Cfinal.to_csv(OUT/"citas_principales_final.csv", index=False)

# Parte 4 · Descriptivos y muestra de validación de relevancia
**Qué hace esta parte:** describe el corpus (procedencia, tipo de documento, tamaño de equipo, periodo) y revisa si los artículos realmente tratan de orientación emprendedora y de mercado, midiendo si esos términos aparecen en el título o las palabras clave o solo en el resumen. Genera `tabla1_descriptiva.csv` y una muestra para validar la relevancia a mano.

Nota: la variable `procedencia` de la primera celda se calcula con `sources_merged`; la versión definitiva (a partir de las banderas `ismain_wos` e `ismain_scopus`) se reconstruye en la Parte 6 (celda R), y las tablas finales salen de la celda T.


In [ ]:
import pandas as pd, numpy as np, re, os
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

D   = '/content/drive/MyDrive/BibFusion/trabajo_limpio/'
OUT = D + 'descriptivos/'; os.makedirs(OUT, exist_ok=True)
M = pd.read_csv(D+'corpus_principal_442.csv', low_memory=False)
E = pd.read_csv(D+'equipos_personas_por_articulo.csv', low_memory=False)
C = pd.read_csv(D+'citas_principales_final.csv', low_memory=False)
for n, d in [('corpus', M), ('equipos', E), ('citas', C)]:
    print(n, d.shape); print(list(d.columns), '\n')

def col(df, *pats):
    for p in pats:
        for c in df.columns:
            if re.fullmatch(p, c, re.I): return c
    for p in pats:
        for c in df.columns:
            if re.search(p, c, re.I): return c
    return None

c_year = col(M, r'year', r'publication_year', r'^py$')
c_type = col(M, r'document_type', r'doc_type', r'^type$')
c_src  = col(M, r'source_title', r'journal', r'^so$')
c_prov = col(M, r'sources_merged')
print('Detectadas ->', dict(anio=c_year, tipo=c_type, revista=c_src, procedencia=c_prov))
assert c_year and c_prov, 'Falta año o procedencia: pásame la lista de columnas'

M = M.copy()
M[c_year] = pd.to_numeric(M[c_year], errors='coerce')
p = M[c_prov].astype(str).str.lower()
hw, hs = p.str.contains('wos|web of science'), p.str.contains('scopus')
M['procedencia'] = np.select([hw & hs, hw, hs], ['Ambas', 'Solo WoS', 'Solo Scopus'], 'Sin dato')
print('\nValores de sources_merged:\n', M[c_prov].value_counts().head(10))

# Llave común para cruzar con equipos y citas
def llave(df):
    for k in ['SR', 'sr', '__doi_norm', 'doi', 'DOI']:
        if k in df.columns and k in M.columns: return k
kE, kC = llave(E), llave(C)
print('\nLlave equipos:', kE, '| llave citas:', kC)

# Tamaño de equipo = personas únicas por artículo
cP = col(E, r'personid', r'person_id', r'persona')
if kE and cP:
    ts = E.drop_duplicates([kE, cP]).groupby(kE).size().rename('tam_equipo')
    M = M.merge(ts, left_on=kE, right_index=True, how='left')
else:
    print('No pude calcular tamaño de equipo: revisa kE/cP')

# Referencias por artículo
if kC:
    nref = C.groupby(kC).size().rename('n_refs')
    M = M.merge(nref, left_on=kC, right_index=True, how='left')

# ---------- Tabla 1 (formato largo) ----------
filas = []
def bloque(nombre, serie):
    vc = serie.fillna('Sin dato').value_counts()
    for k, v in vc.items():
        filas.append([nombre, k, int(v), round(100*v/len(serie), 1)])
bloque('Procedencia', M['procedencia'])
if c_type: bloque('Tipo de documento', M[c_type])
if 'tam_equipo' in M:
    cat = pd.cut(M['tam_equipo'], [0, 1, 2, 3, 4, 999], labels=['1 (autor único)', '2', '3', '4', '5+'])
    bloque('Tamaño de equipo', cat.astype(object))
bloque('Período', pd.cut(M[c_year], [1999, 2009, 2015, 2020, 2026],
       labels=['2000-2009', '2010-2015', '2016-2020', '2021-2026']).astype(object))
T1 = pd.DataFrame(filas, columns=['Variable', 'Categoría', 'n', '%'])
T1.to_csv(OUT+'tabla1_descriptiva.csv', index=False); print('\n', T1.to_string(index=False))

# ---------- Figuras ----------
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})

pv = M.groupby([c_year, 'procedencia']).size().unstack(fill_value=0)
ax = pv.plot(kind='bar', stacked=True, figsize=(10, 4)); ax.set_xlabel('Año'); ax.set_ylabel('Artículos')
ax.set_xticklabels([str(int(x)) for x in pv.index], rotation=90)
plt.tight_layout(); plt.savefig(OUT+'fig1_produccion_anual.png', dpi=300); plt.show()

if 'tam_equipo' in M:
    ax = M['tam_equipo'].clip(upper=8).value_counts().sort_index().plot(kind='bar', figsize=(5, 3.5))
    ax.set_xlabel('Autores por artículo (8 = 8 o más)'); ax.set_ylabel('Artículos')
    plt.tight_layout(); plt.savefig(OUT+'fig2_tamano_equipo.png', dpi=300); plt.show()

if c_src:
    top = M[c_src].astype(str).str.upper().value_counts().head(15)[::-1]
    ax = top.plot(kind='barh', figsize=(7, 5)); ax.set_xlabel('Artículos')
    plt.tight_layout(); plt.savefig(OUT+'fig3_revistas_top15.png', dpi=300); plt.show()

# ---------- Cobertura de referencias ----------
if 'n_refs' in M:
    cob = M.groupby('procedencia')['n_refs'].agg(['count', 'median', 'min', 'max'])
    cob['sin_refs'] = M[M['n_refs'].isna()].groupby('procedencia').size()
    print('\nReferencias por artículo y procedencia:\n', cob.fillna(0))
    cob.to_csv(OUT+'cobertura_referencias_procedencia.csv')
    por_anio = M.groupby(c_year)['n_refs'].median()
    ax = por_anio.plot(figsize=(8, 3.5), marker='o'); ax.set_ylabel('Mediana de referencias'); ax.set_xlabel('Año')
    plt.tight_layout(); plt.savefig(OUT+'fig4_referencias_por_anio.png', dpi=300); plt.show()
print('\nGuardado en', OUT)

In [ ]:
# Variables auxiliares que usa la celda de validación (txt, cb, revista, cc)
cb_col = next((c for c in ['cited_by', 'cited_by_count', 'times_cited'] if c in M.columns), None)
M['cb'] = pd.to_numeric(M[cb_col], errors='coerce').fillna(0) if cb_col else 0
M['revista'] = M[c_src] if c_src else ''
txt = (M['title'].fillna('') + ' ' + M['abstract'].fillna('') + ' ' + M['author_keywords'].fillna('')).str.lower()
cc = M['country'].fillna('').apply(lambda s: {x.strip() for x in str(s).split(';') if x.strip()}) if 'country' in M.columns else None
print('Columna de citas usada:', cb_col, '| artículos:', len(M), '| con resumen:', int(M['abstract'].fillna('').ne('').sum()))


In [ ]:
# Intensidad: ¿están EO y MO en el título/palabras clave, o solo en el resumen?
campo_tk = (M['title'].fillna('') + ' ' + M['author_keywords'].fillna('')).str.lower()
EO_re = r'entrepreneur\w*[\s\-]+(?:orient|posture|strategic orient)|entrepreneurially[\s\-]orient|\beo\b'
MO_re = r'market[\s\-]+orient|marketing[\s\-]+orient|customer[\s\-]+orient'
M['eo_tk'] = campo_tk.str.contains(EO_re); M['mo_tk'] = campo_tk.str.contains(MO_re)
M['n_menciones'] = txt.str.count(EO_re) + txt.str.count(MO_re)
M['nucleo_tk'] = np.select([M.eo_tk & M.mo_tk, M.eo_tk | M.mo_tk],
                           ['EO y MO en título/palabras clave', 'Solo uno de los dos'], 'Solo en resumen')
print(pd.crosstab(M['procedencia'], M['nucleo_tk'], margins=True))
print(M.groupby('procedencia')['n_menciones'].describe())
for u in [1, 2, 3, 5]:
    print(f'Artículos con <={u} menciones:', (M['n_menciones'] <= u).sum())

# Muestra de validación rehecha: "ambiguo" = EO y MO no están ambos en título/palabras clave, menos menciones primero
usados = set()
def toma(df, n, etiqueta):
    d = df[~df['SR'].isin(usados)].head(n).copy(); d['estrato'] = etiqueta; usados.update(d['SR']); return d
amb = M[~(M.eo_tk & M.mo_tk)].sort_values('n_menciones')
S = pd.concat([
    toma(amb, 20, 'ambiguo/limite'),
    toma(M.sort_values('cb', ascending=False), 20, 'altamente citado'),
    toma(M[M['year'] >= 2025].sample(frac=1, random_state=14), 20, 'reciente'),
    toma(M.sample(frac=1, random_state=14), 30, 'aleatorio'),
])
S = S[['estrato', 'SR', 'year', 'procedencia', 'revista', 'document_type', 'n_menciones', 'title', 'author_keywords', 'abstract']]
for c in ['relevancia_estudiante', 'relevancia_profesor', 'nota']: S[c] = ''
S.to_csv(OUT+'muestra_validacion_relevancia.csv', index=False, encoding='utf-8-sig')
print(S['estrato'].value_counts())

# Parte 5 · OpenAlex: citas, historia de autores y base de modelos
**Qué hace esta parte:** (1) empareja cada artículo con OpenAlex por DOI o por título; (2) descarga las citas por año de publicación, que permiten medir las citas a tres años; (3) descarga la historia previa de cada autor (obras, coautores, países, instituciones y campo); (4) calcula las variables del equipo (familiaridad, diversidad, liderazgo, nuevos en el campo) usando solo información hasta t−1; (5) descarga la citación media de cada revista; (6) arma `base_modelo_442.csv`.

Guarda `openalex_match_442.csv`, `openalex_citas_por_anio.csv`, `resultado_citas_442.csv`, `historia_autores_openalex.csv`, `personid_openalex.csv`, `base_modelo_442.csv` y `openalex_revistas_442.csv`. Requiere la clave de OpenAlex, que se pide de forma oculta (no la escribas en las celdas). Esta parte puede tardar horas; si se interrumpe, vuelve a ejecutarla: los resultados guardados en Drive se reutilizan.


In [ ]:
import requests, time, getpass, difflib, re
KEY  = getpass.getpass('OpenAlex API key (Enter si no usas): ')
MAIL = input('mailto opcional (Enter para omitir): ').strip()

def oa(path, **p):
    if KEY: p['api_key'] = KEY
    if MAIL: p['mailto'] = MAIL
    for i in range(4):
        try:
            r = requests.get('https://api.openalex.org'+path, params=p, timeout=40)
        except Exception:
            time.sleep(2*(i+1)); continue
        if r.status_code == 200: return r.json()
        if r.status_code == 404: return None
        time.sleep(2*(i+1))
    return None

norm = lambda s: re.sub(r'[^a-z0-9 ]', '', str(s).lower())
def buscar(row):
    doi = row['__doi_norm']
    if isinstance(doi, str) and doi:
        w = oa('/works/doi:'+doi)
        if w: return w, 'doi'
    w = oa('/works', search=str(row['title'])[:200], filter=f"publication_year:{int(row['year'])}", **{'per-page': 3})
    for c in (w or {}).get('results', []):
        if difflib.SequenceMatcher(None, norm(c.get('title')), norm(row['title'])).ratio() >= 0.9:
            return c, 'titulo'
    return None, 'sin_match'

res, largo = [], []
for k, (_, row) in enumerate(M.iterrows(), 1):
    w, how = buscar(row)
    if w:
        wid = w['id'].split('/')[-1]
        g = oa('/works', filter=f'cites:{wid}', group_by='publication_year') or {}
        for it in g.get('group_by', []):
            largo.append((row['SR'], int(it['key']), it['count']))
        res.append((row['SR'], wid, how, w.get('publication_year'), w.get('cited_by_count')))
    else:
        res.append((row['SR'], None, how, None, None))
    if k % 50 == 0: print(k, 'de', len(M))

R_oa = pd.DataFrame(res, columns=['SR', 'oa_id', 'match', 'oa_year', 'oa_cited_by'])
L_oa = pd.DataFrame(largo, columns=['SR', 'anio_cita', 'n'])
R_oa.to_csv(D+'openalex_match_442.csv', index=False); L_oa.to_csv(D+'openalex_citas_por_anio.csv', index=False)
print(R_oa['match'].value_counts())
print('Año OpenAlex distinto del corpus:', (R_oa.merge(M[['SR','year']], on='SR').eval('oa_year != year')).sum())

In [ ]:
Z = R_oa.merge(M[['SR', 'year', 'document_type', 'procedencia', '__doi_norm']], on='SR')
Z['dif'] = Z['oa_year'] - Z['year']
print('Diferencia de año (OpenAlex - corpus):'); print(Z['dif'].value_counts(dropna=False).sort_index())

# Control: suma de citas por año vs total de OpenAlex
s = L_oa.groupby('SR')['n'].sum().rename('suma_anios')
Z = Z.merge(s, left_on='SR', right_index=True, how='left')
print('\n|suma por año - cited_by OpenAlex|:'); print((Z['suma_anios'].fillna(0) - Z['oa_cited_by']).abs().describe())

print('\nSin match:'); print(Z[Z['match'] == 'sin_match'][['SR', 'year', 'document_type', 'procedencia', '__doi_norm']].to_string())

# Año ancla seguro y ventanas de citas
Z['t'] = np.where(Z['oa_year'].notna(), np.minimum(Z['year'], Z['oa_year']), Z['year']).astype(int)
Lw = L_oa.merge(Z[['SR', 't', 'year']], on='SR')
def cita(ancla, a, b):
    m = (Lw['anio_cita'] >= Lw[ancla] + a) & (Lw['anio_cita'] <= Lw[ancla] + b)
    return Lw[m].groupby('SR')['n'].sum()
for nom, (anc, a, b) in {'c3': ('t', 1, 3), 'c3_corpus': ('year', 1, 3), 'c2': ('t', 1, 2)}.items():
    Z[nom] = Z['SR'].map(cita(anc, a, b)).fillna(0)
    Z.loc[Z['oa_id'].isna(), nom] = np.nan
Z['censurado3'] = Z['t'] + 3 > 2025

ok = Z[(~Z['censurado3']) & Z['c3'].notna()]
print('\nCon ventana de 3 años completa y match:', len(ok), '| censurados:', int(Z['censurado3'].sum()))
print(ok['c3'].describe()); print('Umbral top decil:', ok['c3'].quantile(.9), '| % con 0 citas:', round((ok['c3'] == 0).mean()*100, 1))
print('Corr c3 vs c3_corpus:', round(ok[['c3', 'c3_corpus']].corr().iloc[0, 1], 3))
print('Sin match dentro de los no censurados:', int((~Z['censurado3'] & Z['oa_id'].isna()).sum()))
Z.to_csv(D+'resultado_citas_442.csv', index=False)

In [ ]:
def limpia(t): return re.split(r'\s*;?\s*\[', str(t))[0].strip()
nm = Z[Z['match'] == 'sin_match'][['SR', 'year']].merge(M[['SR', 'title']], on='SR')
resc = []
for _, row in nm.iterrows():
    t, y = limpia(row['title']), int(row['year'])
    w = oa('/works', search=t[:200], filter=f'from_publication_date:{y-3}-01-01,to_publication_date:{y+1}-12-31', **{'per-page': 5})
    mejor, sc = None, 0
    for c in (w or {}).get('results', []):
        r = difflib.SequenceMatcher(None, norm(c.get('title')), norm(t)).ratio()
        if r > sc: mejor, sc = c, r
    resc.append((row['SR'], mejor['id'].split('/')[-1] if mejor else None, round(sc, 2),
                 mejor.get('publication_year') if mejor else None, mejor.get('cited_by_count') if mejor else None,
                 (mejor.get('title') or '')[:70] if mejor else None, t[:70]))
Rr = pd.DataFrame(resc, columns=['SR', 'oa_id', 'sim', 'oa_year', 'oa_cited_by', 'titulo_oa', 'titulo_corpus'])
print(Rr.sort_values('sim', ascending=False).to_string())

# Aceptar automáticamente sim >= 0.90 y actualizar tablas
for r in Rr[Rr['sim'] >= 0.90].itertuples():
    g = oa('/works', filter=f'cites:{r.oa_id}', group_by='publication_year') or {}
    L_oa = pd.concat([L_oa, pd.DataFrame([(r.SR, int(i['key']), i['count']) for i in g.get('group_by', [])],
                                         columns=['SR', 'anio_cita', 'n'])], ignore_index=True)
    R_oa.loc[R_oa['SR'] == r.SR, ['oa_id', 'match', 'oa_year', 'oa_cited_by']] = [r.oa_id, 'titulo_rescate', r.oa_year, r.oa_cited_by]
R_oa.to_csv(D+'openalex_match_442.csv', index=False); L_oa.to_csv(D+'openalex_citas_por_anio.csv', index=False)
print('Aceptados:', (Rr['sim'] >= 0.90).sum(), '| para revisar a mano (0.80-0.90):', ((Rr['sim'] >= .8) & (Rr['sim'] < .9)).sum())

**Definiciones que necesita la siguiente celda.** La descarga de la historia de autores usa `CAMPOS` (campos que se piden a OpenAlex) y `P` (personas con identificador: `PersonID`, `oa_author`, `orcid`). Aquí se construyen a partir de `T2` (Parte 2). Si ya tienes `historia_autores_openalex.csv` en `trabajo_limpio/`, la celda siguiente solo descarga los autores que falten.


In [ ]:
CAMPOS = 'id,publication_year,type,authorships,primary_topic'
P = T2[['PersonID']].drop_duplicates().copy()
P['oa_author'] = P['PersonID'].where(P['PersonID'].str.startswith('OA:')).str.replace('OA:', '', regex=False)
P['orcid']     = P['PersonID'].where(P['PersonID'].str.startswith('ORCID:')).str.replace('ORCID:', '', regex=False)
P = P[P['oa_author'].notna() | P['orcid'].notna()].reset_index(drop=True)
P = P.astype(object).where(P.notna(), None)
print('Personas con identificador:', len(P), '(el artículo reporta 1.048 de 1.199)')


In [ ]:
def sid(x): return (x or '').split('/')[-1]

def obras(aid):
    out, cur = [], '*'
    for _ in range(10):
        j = oa('/works', filter=f'author.id:{aid}', select=CAMPOS, **{'per-page': 200, 'cursor': cur})
        if not j:
            if cur == '*': raise RuntimeError('sin respuesta en la primera página')
            break
        for w in j['results']:
            au = w.get('authorships') or []; tp = w.get('primary_topic') or {}
            pos = next((a.get('author_position') for a in au if sid((a.get('author') or {}).get('id')) == aid), None)
            out.append(dict(autor=aid, work=sid(w.get('id')), year=w.get('publication_year'), tipo=w.get('type'),
                n_aut=len(au), pos=pos,
                coaut=';'.join(sid((a.get('author') or {}).get('id')) for a in au),
                paises=';'.join(sorted({c for a in au for c in (a.get('countries') or [])})),
                inst=';'.join(sorted({sid(i.get('id')) for a in au for i in (a.get('institutions') or []) if i.get('id')})),
                campo=(tp.get('field') or {}).get('display_name'), topico=sid(tp.get('id'))))
        cur = (j.get('meta') or {}).get('next_cursor')
        if not cur: break
    return out

hist_path = D+'historia_autores_openalex.csv'
hecho = set(pd.read_csv(hist_path)['autor']) if os.path.exists(hist_path) else set()
buf, res_aut, fallos = [], [], []
for k, r in enumerate(P.itertuples(), 1):
    aid = r.oa_author
    if not aid and r.orcid:
        j = oa('/authors/orcid:'+r.orcid); aid = sid(j['id']) if j else None
    res_aut.append((r.PersonID, aid))
    if aid and aid not in hecho:
        try:
            buf += obras(aid); hecho.add(aid)
        except Exception as e:
            fallos.append((aid, repr(e)))
    if k % 100 == 0 or k == len(P):
        if buf:
            pd.DataFrame(buf).to_csv(hist_path, mode='a', header=not os.path.exists(hist_path), index=False); buf = []
        print(k, 'de', len(P), '| fallos:', len(fallos))
pd.DataFrame(res_aut, columns=['PersonID', 'oa_author']).to_csv(D+'personid_openalex.csv', index=False)
pd.DataFrame(fallos, columns=['autor', 'error']).to_csv(D+'historia_fallos.csv', index=False)
H = pd.read_csv(hist_path)
print('Autores con historia:', H['autor'].nunique(), '| trabajos:', len(H)); print(H.groupby('autor').size().describe())

In [ ]:
import itertools
PI = pd.read_csv(D+'personid_openalex.csv')
H = pd.read_csv(hist_path).dropna(subset=['year']); H['year'] = H['year'].astype(int)
H['coaut_set'] = H['coaut'].fillna('').str.split(';').apply(set)
H['topico'] = H['topico'].fillna(''); H['campo'] = H['campo'].fillna('')
print(H['campo'].value_counts().head(8))
EMPTY = H.iloc[0:0]; HA = {a: g.sort_values('year') for a, g in H.groupby('autor')}
CAMPO_EO = 'Business, Management and Accounting'      # campo de referencia para "nuevo en el campo"
T = E.drop_duplicates(['SR', 'PersonID']).merge(PI.drop_duplicates('PersonID'), on='PersonID', how='left')
base = Z[['SR', 't', 'c3', 'c2', 'c3_corpus', 'censurado3', 'oa_id', 'match']].copy()

def pasado(a, t, w):
    g = HA.get(a, EMPTY); return g[(g['year'] <= t-1) & (g['year'] >= t-w)]

def familiaridad(ids_, t, w):
    if len(ids_) < 2: return (np.nan, np.nan, np.nan)
    pas = {a: pasado(a, t, w) for a in ids_}; cnt = []
    for i, j in itertools.combinations(ids_, 2):
        gi, gj = pas[i], pas[j]
        wi = set(gi.loc[gi['coaut_set'].apply(lambda s: j in s) & (gi['n_aut'] <= 25), 'work'])
        wj = set(gj.loc[gj['coaut_set'].apply(lambda s: i in s) & (gj['n_aut'] <= 25), 'work'])
        cnt.append(len(wi | wj))
    cnt = np.array(cnt); return ((cnt > 0).mean(), cnt.mean(), cnt.max())

def dist_tema(ids_, t, col, w=5):
    S = [set(x for x in pasado(a, t, w)[col] if x) for a in ids_]
    d = [1 - len(x & y)/len(x | y) for x, y in itertools.combinations(S, 2) if (x | y)]
    return (np.mean(d) if d else np.nan, len(set().union(*S)) if S else 0)

filas, MEM = [], []
for r in base.itertuples():
    mem = T[T['SR'] == r.SR]; ids_ = list(mem['oa_author'].dropna().unique())
    f = dict(SR=r.SR, n_miembros=len(mem), n_id=len(ids_), cobertura_id=len(ids_)/max(len(mem), 1))
    for w, nom in [(5, 'w5'), (3, 'w3'), (7, 'w7'), (99, 'wfull')]:
        f[f'fam_dens_{nom}'], f[f'fam_rec_{nom}'], f[f'fam_max_{nom}'] = familiaridad(ids_, r.t, w)
    f['div_tema'], f['n_topicos'] = dist_tema(ids_, r.t, 'topico')
    f['div_campo'], f['n_campos'] = dist_tema(ids_, r.t, 'campo')
    for a in ids_:
        pr = pasado(a, r.t, 99)
        MEM.append(dict(SR=r.SR, a=a, n_prev=len(pr), n_first=int((pr['pos'] == 'first').sum()),
                        antig=(r.t - pr['year'].min()) if len(pr) else 0, campo_prev=int((pr['campo'] == CAMPO_EO).sum())))
    filas.append(f)
F = pd.DataFrame(filas); MEMd = pd.DataFrame(MEM)

# Liderazgo y nuevos integrantes (umbrales relativos: P50 flexible, P75 moderado, P90 restrictivo)
q = MEMd['n_prev'].quantile([.5, .75, .9]).values; print('Umbrales de trabajos previos (P50, P75, P90):', q)
MEMd['lid_flex'] = MEMd['n_prev'] >= q[0]; MEMd['lid_mod'] = MEMd['n_prev'] >= q[1]; MEMd['lid_res'] = MEMd['n_prev'] >= q[2]
MEMd['nuevo_campo'] = MEMd['campo_prev'] == 0; MEMd['junior'] = MEMd['antig'] <= 5
g = MEMd.groupby('SR')
L = pd.DataFrame({'lid_flex_n': g['lid_flex'].sum(), 'lid_mod_n': g['lid_mod'].sum(), 'lid_res_n': g['lid_res'].sum(),
    'nuevos_campo_n': g['nuevo_campo'].sum(), 'junior_n': g['junior'].sum(),
    'exp_media': g['n_prev'].mean(), 'exp_max': g['n_prev'].max(), 'exp_sd': g['n_prev'].std(), 'antig_media': g['antig'].mean(),
    'conc_max': g['n_prev'].apply(lambda s: s.max()/s.sum() if s.sum() > 0 else np.nan),
    'hhi': g['n_prev'].apply(lambda s: ((s/s.sum())**2).sum() if s.sum() > 0 else np.nan)}).reset_index()
L['nuevo_con_lider'] = (L['nuevos_campo_n'] > 0) & (L['lid_mod_n'] > 0)

# Variables del artículo focal (no usan información futura): diversidad geográfica/institucional, controles
def blau(l): p = pd.Series(l).value_counts(normalize=True); return 1 - (p**2).sum()
paises = M['country'].fillna('').apply(lambda s: [x.strip() for x in s.split(';') if x.strip()])
M2 = pd.DataFrame({'SR': M['SR'], 'year_corpus': M['year'], 'n_refs': M['n_refs'],
    'n_paises': paises.apply(lambda l: len(set(l))), 'blau_pais': paises.apply(lambda l: blau(l) if l else np.nan),
    'n_inst': M['affiliations'].fillna('').apply(lambda s: len({x.strip() for x in s.split(';') if x.strip()})),
    'revision': M['document_type'].str.contains('REVIEW', na=False).astype(int),
    'retractado': M['document_type'].str.contains('RETRACTED', na=False).astype(int),
    'alfabetico': M['author'].apply(lambda a: (lambda n: int(len(n) >= 3 and n == sorted(n)))(
        [x.strip().split(' ')[0].upper() for x in str(a).split(';') if x.strip()]))})

D2 = base.merge(F, on='SR').merge(L, on='SR', how='left').merge(M2, on='SR', how='left')
D2['elegible'] = (~D2['censurado3']) & D2['c3'].notna() & (D2['n_miembros'] >= 2) & (D2['retractado'] == 0)
D2.to_csv(D+'base_modelo_442.csv', index=False)
el = D2[D2['elegible']]
print('N elegible:', len(el), '| con familiaridad medible (>=2 identificados):', int(el['fam_dens_w5'].notna().sum()))
print('Equipos con algún vínculo previo (w5):', round((el['fam_dens_w5'].dropna() > 0).mean()*100, 1), '%')
print(el[['cobertura_id', 'fam_dens_w5', 'fam_rec_w5', 'lid_mod_n', 'nuevos_campo_n', 'n_paises', 'div_tema', 'exp_media']].describe().T)

In [ ]:
# Función auxiliar (copiada de la celda SJR eliminada)
def i8(x):
    s = re.sub(r'[^0-9Xx]', '', str(x).split('.')[0])
    return s.zfill(8).upper() if s else None

corto = lambda x: (x or '').split('/')[-1] or None
guion = lambda s: s[:4] + '-' + s[4:] if s else None
iss = M.set_index('SR')['issn'].apply(i8).to_dict()
fuentes, filas = {}, []
for r in R_oa.itertuples():
    sid = None
    if isinstance(r.oa_id, str):
        w = oa('/works/'+r.oa_id, select='id,primary_location')
        sid = corto((((w or {}).get('primary_location') or {}).get('source') or {}).get('id'))
    if not sid and iss.get(r.SR):
        j = oa('/sources/issn:'+guion(iss[r.SR]), select='id'); sid = corto((j or {}).get('id'))
    if sid and sid not in fuentes:
        fuentes[sid] = oa('/sources/'+sid, select='id,display_name,summary_stats,works_count') or {}
    fu = fuentes.get(sid) or {}; ss = fu.get('summary_stats') or {}
    filas.append((r.SR, sid, fu.get('display_name'), ss.get('2yr_mean_citedness'), ss.get('h_index'), fu.get('works_count')))
Fu = pd.DataFrame(filas, columns=['SR', 'source_id', 'revista_oa', 'citedness2y', 'h_index', 'works_rev'])
Fu.to_csv(D+'openalex_revistas_442.csv', index=False)
print('Cobertura métrica de revista (OpenAlex):', Fu['citedness2y'].notna().sum(), 'de', len(Fu))


# Parte 6 · Análisis final del artículo
Estas celdas leen los CSV de `trabajo_limpio/` y **no llaman a ninguna API**; son las que reproducen los resultados del artículo. **Orden: R → T → K3 → U → V**, todas en la misma sesión.

| Celda | Qué hace | Resultado en el artículo |
|---|---|---|
| **R** | Reconstruye la muestra analítica (N = 257), define las variables y estima los cinco modelos anidados con Poisson y MCO | Tabla 1 y Tablas S4 y S5 |
| **T** | Genera las tablas finales: corpus, descriptivos de la muestra y modelos anidados | Tablas S2, S3 y S4–S5 |
| **K3** | Genera las figuras F2 a F7 (producción, perfil de equipos, efectos ajustados, curva de familiaridad, robustez) | Figuras 2 a 5 y S1 y S2 |
| **U** | Estima el modelo con número de instituciones (N = 241) | Sección de resultados |
| **V** | Calcula cuánto explican los modelos (R², pseudo-R²) y la prueba conjunta del bloque de equipo; no modifica archivos | Tabla S6 y párrafo de poder explicativo |


**Celda R.** Reconstruye la muestra, las variables y los cinco modelos. Debe imprimir el corpus de 442 artículos (Ambas bases 291, solo Scopus 118, solo WoS 33).

In [ ]:
# ===== CELDA R: reconstruye el análisis final desde los CSV guardados en Drive (no llama a ninguna API) =====
import pandas as pd, numpy as np, os, re
import statsmodels.api as sm, statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/BibFusion/trabajo_limpio/'; OUT = D + 'descriptivos/'; os.makedirs(OUT, exist_ok=True)

# 1) Corpus y equipos; procedencia según las banderas (no según sources_merged)
M = pd.read_csv(D + 'corpus_principal_442.csv', low_memory=False)
E = pd.read_csv(D + 'equipos_personas_por_articulo.csv', low_memory=False)
M['year'] = pd.to_numeric(M['year'], errors='coerce'); c_year = 'year'
for k in ['ismain_wos', 'ismain_scopus']: M[k] = M[k].astype(str).str.lower().eq('true')
M['procedencia'] = np.select([M.ismain_wos & M.ismain_scopus, M.ismain_wos, M.ismain_scopus], ['Ambas', 'Solo WoS', 'Solo Scopus'], 'Sin dato')
print('Corpus:', len(M), '| procedencia:', M['procedencia'].value_counts().to_dict(), '(esperado 442 | Ambas 291, Solo Scopus 118, Solo WoS 33)')

# 2) Base de modelos: equipos elegibles y variables derivadas
D2 = pd.read_csv(D + 'base_modelo_442.csv')
D2['elegible'] = D2['elegible'].astype(str).str.lower().eq('true')
A = D2[D2['elegible']].copy()
A['lt'] = np.log(A['n_miembros']); A['lrefs'] = np.log1p(A['n_refs']); A['lexp'] = np.log1p(A['exp_media'])
A['linst'] = np.log1p(A['n_inst']); A['lider_any'] = (A['lid_mod_n'] > 0).astype(int)
A['nuevos_share'] = A['nuevos_campo_n'] / A['n_id'].clip(lower=1)
A['periodo'] = pd.cut(A['t'], [1999, 2012, 2016, 2019, 2022], labels=['<=2012', '2013-16', '2017-19', '2020-22'])
req = ['fam_dens_w5', 'blau_pais', 'linst', 'div_campo', 'lider_any', 'conc_max', 'nuevos_share', 'lexp', 'lrefs', 'cobertura_id']
A = A.dropna(subset=req + ['c3']).copy()
for c in ['fam_dens_w5', 'blau_pais', 'div_campo', 'conc_max', 'nuevos_share']: A[c + '_c'] = A[c] - A[c].mean()
A['fam2'] = A['fam_dens_w5_c'] ** 2; A['lc3'] = np.log1p(A['c3'])

# 3) Métrica de revista (OpenAlex, ya descargada) y variables finales
Fu = pd.read_csv(D + 'openalex_revistas_442.csv')
A3 = A.merge(M[['SR', 'procedencia']], on='SR', how='left').merge(Fu[['SR', 'citedness2y', 'h_index']], on='SR', how='left')
A3 = A3[A3['citedness2y'].notna()].copy(); A3['lif'] = np.log1p(A3['citedness2y'])
B = A3.copy()
B['una_base'] = (B['procedencia'] != 'Ambas').astype(int); B['inst_ok'] = B['n_inst'] > 0
n_ = B['n_id'].clip(lower=2); B['conc_norm'] = (B['conc_max'] - 1 / n_) / (1 - 1 / n_)
B['multi_pais'] = (B['n_paises'] >= 2).astype(int); B['lif_w'] = B['lif'].clip(upper=B['lif'].quantile(.99))

def ajusta(d, f, est):
    d = d.assign(lc3=np.log1p(d['c3']))
    return (smf.ols('lc3 ~ ' + f, d).fit(cov_type='HC3') if est == 'OLS'
            else smf.glm('c3 ~ ' + f, d, family=sm.families.Poisson()).fit(cov_type='HC1'))

# 4) Comprobaciones: deben coincidir con lo reportado en el manuscrito
d257 = B[B['n_id'] >= 2]; d241 = d257[d257['inst_ok']]; d212 = d241[d241['cobertura_id'] >= 1]
print(f'N modelo A = {len(A)} (≈258) | A3 = {len(A3)} (257) | d257 = {len(d257)} (257) | d241 = {len(d241)} (241) | d212 = {len(d212)} (212)')
print('Media de citas a 3 años en d257:', round(d257['c3'].mean(), 2), '(esperado 19.66) | lif_w máximo:', round(B['lif_w'].max(), 2))


**Celda T.** Guarda las tablas finales (`tabla1_descriptiva`, `tabla2_descriptivos_modelo`, `tabla3_final_ols` y `tabla3_final_poisson`).

In [ ]:
# ===== CELDA T: tablas finales del manuscrito (Tablas 1, 2 y 3). Ejecutar después de la celda R =====
# Tabla 1: corpus de 442 artículos
filas = []
def bloque(nombre, serie):
    vc = serie.fillna('Sin dato').value_counts()
    for k, v in vc.items(): filas.append([nombre, k, int(v), round(100 * v / len(serie), 1)])
bloque('Procedencia', M['procedencia']); bloque('Tipo de documento', M['document_type'])
tam = E.drop_duplicates(['SR', 'PersonID']).groupby('SR').size().rename('tam_equipo')
cat = pd.cut(M[['SR']].merge(tam, left_on='SR', right_index=True, how='left')['tam_equipo'], [0, 1, 2, 3, 4, 999], labels=['1', '2', '3', '4', '5+'])
bloque('Tamaño de equipo', cat.astype(object))
bloque('Período', pd.cut(M['year'], [1999, 2009, 2015, 2020, 2026], labels=['2000-2009', '2010-2015', '2016-2020', '2021-2026']).astype(object))
T1 = pd.DataFrame(filas, columns=['Variable', 'Categoría', 'n', '%']); T1.to_csv(OUT + 'tabla1_descriptiva.csv', index=False); print(T1.to_string(index=False))

# Tabla 2: descriptivos de la muestra analítica y correlaciones de Spearman
vars_ = ['c3', 'n_miembros', 'fam_dens_w5', 'fam_rec_w5', 'n_paises', 'blau_pais', 'n_inst', 'div_campo', 'lid_mod_n', 'conc_max',
         'nuevos_share', 'exp_media', 'citedness2y', 'n_refs']
T2 = A3[vars_].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']].round(2); T2.to_csv(OUT + 'tabla2_descriptivos_modelo.csv'); print('\n', T2)
A3[vars_].corr(method='spearman').round(2).to_csv(OUT + 'tabla2b_correlaciones_spearman.csv')

# Tabla 3: cinco modelos anidados, MCO y Poisson
d257 = B[B['n_id'] >= 2].copy(); d257['conc_norm_c'] = d257['conc_norm'] - d257['conc_norm'].mean()
c0 = 'lt + lexp + lrefs + lif_w + revision + alfabetico + cobertura_id + periodo + una_base'
m2 = c0 + ' + fam_dens_w5_c + fam2'; m3 = m2 + ' + multi_pais + div_campo_c'; m4 = m3 + ' + lider_any + conc_norm_c + nuevos_share_c'
m5 = m4 + ' + fam_dens_w5_c:multi_pais + fam_dens_w5_c:div_campo_c + lider_any:nuevos_share_c'
sp = {'M1 controles': c0, 'M2 +familiaridad': m2, 'M3 +diversidad': m3, 'M4 +liderazgo/nuevos': m4, 'M5 +interacciones': m5}
def tabla3(d, specs, est):
    out = {}
    for nom, f in specs.items():
        r = ajusta(d, f, est); s = pd.DataFrame({'b': r.params, 'se': r.bse, 'p': r.pvalues})
        out[nom] = s.apply(lambda x: f"{x.b:.3f}{'***' if x.p < .01 else '**' if x.p < .05 else '*' if x.p < .1 else ''} ({x.se:.3f})", axis=1)
    T_ = pd.DataFrame(out).fillna('')
    return T_[~T_.index.str.startswith('periodo') & (T_.index != 'Intercept')]
for est in ['OLS', 'Poisson']:
    T_ = tabla3(d257, sp, est); print(f'\n{est} (N={len(d257)})'); print(T_.to_string()); T_.to_csv(OUT + f'tabla3_final_{est.lower()}.csv')


**Celda K3.** Dibuja y guarda las figuras F2 a F7 a 300 dpi.

In [ ]:
# ===== CELDA K3: figuras finales para revista (F2–F7), versión 3 (retoques F2 y F4) =====
import numpy as np, pandas as pd, matplotlib.pyplot as plt, matplotlib as mpl
import statsmodels.api as sm, statsmodels.formula.api as smf
TITULOS = False            # los títulos van en el pie de figura del manuscrito, no dentro de la imagen
AZ, NA, AQ, GR, TX, GRID = '#2a78d6', '#eb6834', '#1baf7a', '#8a8985', '#26251f', '#e4e3df'
mpl.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
    'font.size': 8.5, 'axes.titlesize': 9, 'axes.labelsize': 8.5, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#9a998f', 'axes.labelcolor': TX, 'text.color': TX,
    'xtick.color': '#52514e', 'ytick.color': '#52514e', 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': .5, 'axes.axisbelow': True,
    'figure.dpi': 110, 'savefig.dpi': 300, 'pdf.fonttype': 42, 'ps.fonttype': 42})
def guarda(fig, nombre, titulo=None):
    if TITULOS and titulo: fig.suptitle(titulo, x=.01, ha='left', fontweight='bold', y=1.02)
    fig.savefig(OUT + nombre + '.png', bbox_inches='tight', facecolor='white'); fig.savefig(OUT + nombre + '.pdf', bbox_inches='tight', facecolor='white'); plt.show()
def letra(ax, t, y=1.05, x=-.02, ha='right'): ax.text(x, y, t, transform=ax.transAxes, fontweight='bold', fontsize=10, va='bottom', ha=ha)
def pc(v, d=0): return f'{v:.{d}f}'

d257 = B[B['n_id'] >= 2].copy(); d257['conc_norm_c'] = d257['conc_norm'] - d257['conc_norm'].mean()
d241 = d257[d257['inst_ok']].copy(); d212 = d241[d241['cobertura_id'] >= 1].copy()
c0 = 'lt + lexp + lrefs + lif_w + revision + alfabetico + cobertura_id + periodo + una_base'
m4 = c0 + ' + fam_dens_w5_c + fam2 + multi_pais + div_campo_c + lider_any + conc_norm_c + nuevos_share_c'

# ---------- F2. Producción anual y tamaño medio del equipo ----------
tam = E.groupby('SR').size().rename('n_aut')
P = M[['SR', c_year, 'procedencia']].drop_duplicates('SR').merge(tam, on='SR', how='left').rename(columns={c_year: 'anio'})
P = P[P['anio'].between(2000, 2025)]
orden = [p for p in ['Ambas', 'Solo Scopus', 'Solo WoS'] if p in P['procedencia'].unique()]
col = {'Ambas': AZ, 'Solo Scopus': NA, 'Solo WoS': AQ}
tab = P.groupby(['anio', 'procedencia']).size().unstack(fill_value=0).reindex(columns=orden).reindex(range(2000, 2026), fill_value=0)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.2, 5.2), sharex=True, gridspec_kw={'height_ratios': [2.3, 1], 'hspace': .22})
base = np.zeros(len(tab))
for p in orden:
    a1.bar(tab.index, tab[p], bottom=base, color=col[p], width=.78, label={'Ambas': 'Both databases', 'Solo Scopus': 'Scopus only', 'Solo WoS': 'WoS only'}[p], edgecolor='white', linewidth=.6); base += tab[p].values
a1.set_ylim(0, base.max() * 1.1); a1.set_ylabel('Articles per year')
a1.legend(frameon=False, ncol=3, loc='lower left', bbox_to_anchor=(0, 1.0), title='Indexed in', title_fontsize=8, alignment='left'); letra(a1, 'A')
n_ = P.groupby('anio').size().reindex(range(2000, 2026), fill_value=0)
m_ = P.groupby('anio')['n_aut'].mean().reindex(range(2000, 2026)).where(n_ >= 5)
a2.plot(m_.index, m_.values, color=TX, lw=1.5, marker='o', ms=4, mfc='white'); a2.set_ylabel('Authors per article\n(mean)')
a2.set_xlabel('Publication year'); a2.set_ylim(1.5, max(4.2, np.nanmax(m_.values) + .3)); letra(a2, 'B', 1.02, x=.0, ha='left')
a2.text(.01, .04, 'Years with fewer than 5 articles omitted', transform=a2.transAxes, fontsize=7.5, color='#52514e')
guarda(fig, 'F2_produccion_equipos', 'Annual output and team size')

# ---------- F3. Perfil de los equipos ----------
paneles = [('fam_dens_w5', 'Prior familiarity\n(density of tied pairs)', None),
           ('n_paises', 'Countries per article', 'cnt'), ('div_campo', 'Field diversity\n(0 = same, 1 = different)', None),
           ('lid_mod_n', 'Leaders per team\n(authors at P75)', 'cnt'), ('nuevos_share', 'Share of members\nnew to the field', None)]
paneles = [(c, t, k) for c, t, k in paneles if c in d257.columns]
fig, axs = plt.subplots(1, len(paneles), figsize=(7.4, 2.5)); axs = np.atleast_1d(axs)
for i, (ax, (c, t, k)) in enumerate(zip(axs, paneles)):
    v = d257[c].dropna()
    if k == 'cnt':
        vc = v.round().clip(upper=5).value_counts().sort_index(); ax.bar(vc.index, vc.values / len(v) * 100, color=AZ, width=.7, edgecolor='white')
        ax.set_xticks(vc.index); ax.set_xticklabels([str(int(j)) if j < 5 else '5+' for j in vc.index])
    else:
        ax.hist(v, bins=10, range=(0, 1), color=AZ, edgecolor='white', weights=np.ones(len(v)) / len(v) * 100)
    ax.set_xlabel(t); ax.set_title(f'median {pc(v.median(), 2)}', fontsize=7.5, color=NA, loc='right', pad=2); letra(ax, 'ABCDE'[i])
axs[0].set_ylabel('% of articles')
fig.tight_layout(); guarda(fig, 'F3_perfil_equipos', f'Team profile (N = {len(d257)})')

# ---------- F4. Citas y calidad de la revista según cada característica (sin ajustar) ----------
rng = np.random.default_rng(7)
def ic(y, n=2000):
    y = np.asarray(y, float); mm = [rng.choice(y, len(y)).mean() for _ in range(n)]; return y.mean(), np.percentile(mm, 2.5), np.percentile(mm, 97.5)
d = d257.copy()
d['g_rev'] = pd.qcut(d['lif_w'], 3, labels=['Low', 'Mid', 'High'])
d['g_pais'] = pd.Categorical(np.where(d['multi_pais'] == 1, '2+', '1'), categories=['1', '2+'])
d['g_fam'] = pd.cut(d['fam_dens_w5'], [-.01, 0, .5, 1.01], labels=['None', 'Some', 'Most'])
d['g_conc'] = pd.qcut(d['conc_norm'].rank(method='first'), 3, labels=['Low', 'Mid', 'High'])
d['g_lid'] = pd.Categorical(np.where(d['lider_any'] == 1, 'Yes', 'No'), categories=['No', 'Yes'])
d['g_new'] = pd.Categorical(np.where(d['nuevos_share_c'] > d['nuevos_share_c'].min() + 1e-9, 'Yes', 'No'), categories=['No', 'Yes'])
gs = [('g_rev', 'Journal\nquality'), ('g_pais', 'Countries\nper article'), ('g_fam', 'Prior\nfamiliarity'),
      ('g_conc', 'Leadership\nconcentration'), ('g_lid', 'Leader\npresent'), ('g_new', 'New\nmembers')]
fig, axs = plt.subplots(2, 6, figsize=(8.6, 4.6), sharey=False, gridspec_kw={'wspace': .42, 'hspace': .55})
lims = {0: [np.inf, -np.inf], 1: [np.inf, -np.inf]}
for j, (g, t) in enumerate(gs):
    cats = [k for k in d[g].cat.categories if (d[g] == k).any()]
    for r, (var, c0_) in enumerate([('c3', AZ if j == 0 else NA), ('lif_w', GR)]):
        ax = axs[r, j]
        if r == 1 and g == 'g_rev': ax.axis('off'); continue
        for i, k in enumerate(cats):
            y = d.loc[d[g] == k, var]; m_, lo, hi = ic(y); lims[r] = [min(lims[r][0], lo), max(lims[r][1], hi)]
            ax.errorbar(i, m_, yerr=[[m_ - lo], [hi - m_]], fmt='o', color=c0_, ms=5.5, capsize=2.5, lw=1.3, mec='white', mew=.8)
        ax.set_xlim(-.6, len(cats) - .4); ax.set_xticks(range(len(cats))); ax.set_xticklabels([f'{k}\n{int((d[g] == k).sum())}' for k in cats], fontsize=6.8)
        if r == 0: ax.set_title(t, fontsize=8.5)
for j in range(6):
    axs[0, j].set_ylim(0, lims[0][1] * 1.08)
    if j: axs[0, j].tick_params(labelleft=False)
    if j > 1: axs[1, j].tick_params(labelleft=False)
    if j: axs[1, j].set_ylim(lims[1][0] - .1, lims[1][1] + .1)
axs[1, 1].set_ylim(lims[1][0] - .1, lims[1][1] + .1)
for j in range(2, 6): axs[1, j].sharey(axs[1, 1])
axs[0, 0].set_ylabel('3-year citations\n(mean and 95% CI)')
axs[1, 1].set_ylabel('Journal quality\n(mean and 95% CI)'); fig.text(.5, -.02, 'Numbers under each label: articles per group. Prior familiarity: None = no tied pairs; Some = up to half; Most = more than half', ha='center', fontsize=7.5, color='#52514e')
letra(axs[0, 0], 'A'); letra(axs[1, 1], 'B')
guarda(fig, 'F4_bruto_citas_y_revista', 'Citations and journal quality by team characteristic (unadjusted)')

# ---------- F5. Efectos ajustados (razón de citas esperadas, escala logarítmica) ----------
nom = {'lif_w': 'Journal quality (+1 SD)', 'una_base': 'Indexed in one database only', 'revision': 'Review article',
       'multi_pais': 'Multinational team', 'div_campo_c': 'Field diversity (+1 SD)', 'lider_any': 'Leader present',
       'conc_norm_c': 'Leadership concentration (+1 SD)', 'nuevos_share_c': 'New members (+1 SD)'}
grupo = {'lif_w': 'Context', 'una_base': 'Context', 'revision': 'Context', 'multi_pais': 'Diversity', 'div_campo_c': 'Diversity',
         'lider_any': 'Leadership and renewal', 'conc_norm_c': 'Leadership and renewal', 'nuevos_share_c': 'Leadership and renewal'}
sd = {k: d257[k].std() for k in ['lif_w', 'div_campo_c', 'conc_norm_c', 'nuevos_share_c']}
def efectos(est, datos, f):
    r = ajusta(datos, f, est); ci = r.conf_int(); out = {}
    for k in nom:
        if k in r.params:
            s = sd.get(k, 1); out[k] = [np.exp(v * s) for v in (r.params[k], ci.loc[k, 0], ci.loc[k, 1])] + [r.pvalues[k]]
    return out
EP = efectos('Poisson', d257, m4); claves = [k for k in nom if k in EP]
fig, ax = plt.subplots(figsize=(7.2, 4.2)); y = 0; yt, yl, sep = [], [], []
for i, k in enumerate(claves):
    if i and grupo[k] != grupo[claves[i - 1]]: y += .6; sep.append(y - .55)
    b, lo, hi, p = EP[k]; ax.errorbar(b, -y, xerr=[[b - lo], [hi - b]], fmt='o', color=AZ, mfc=AZ if p < .05 else 'white', mec=AZ, ms=6.5, capsize=2.5, lw=1.4)
    ax.text(3.0, -y, f'{pc(b, 2)} ({pc(lo, 2)}–{pc(hi, 2)})', va='center', fontsize=7.5, color='#52514e'); yt.append(-y); yl.append(nom[k]); y += 1
ax.set_xscale('log'); ax.set_xlim(.2, 2.9); ax.set_xticks([.25, .5, 1, 2]); ax.set_xticklabels(['0.25', '0.5', '1', '2']); ax.minorticks_off()
ax.axvline(1, color=TX, lw=1)
for g_, yy in {grupo[k]: yt[i] for i, k in reversed(list(enumerate(claves)))}.items(): ax.text(.205, yy + .42, g_, fontsize=7.5, style='italic', color='#52514e', va='bottom')
ax.set_yticks(yt); ax.set_yticklabels(yl); ax.grid(axis='y', visible=False)
ax.text(3.0, 1.0, 'Ratio (95% CI)', fontsize=7.5, color='#52514e', va='center')
ax.set_xlabel('Ratio of expected 3-year citations (log scale; 1 = no effect)\nFilled marker: p < 0.05; open: p ≥ 0.05')
fig.tight_layout(); guarda(fig, 'F5_efectos_ajustados', f'Adjusted effects, Poisson model (N = {len(d257)})')

# ---------- F6. Familiaridad y citas ----------
mA = d257['fam_dens_w5'].mean(); x = np.linspace(0, 1, 41)
cont = ['lt', 'lexp', 'lrefs', 'lif_w', 'revision', 'alfabetico', 'cobertura_id', 'una_base', 'multi_pais', 'div_campo_c', 'lider_any', 'conc_norm_c', 'nuevos_share_c']
fig, (ax, axh) = plt.subplots(2, 1, figsize=(5.4, 4.6), sharex=True, gridspec_kw={'height_ratios': [4, 1], 'hspace': .06})
for dd, etq, c in [(d257, f'All teams (N = {len(d257)})', AZ), (d212, f'Fully identified teams (N = {len(d212)})', NA)]:
    f = m4 if dd is d257 else m4.replace(' + cobertura_id', ''); cc = [k for k in cont if k in f]
    r = ajusta(dd, f, 'Poisson'); g = pd.DataFrame({k: [dd[k].mean()] * len(x) for k in cc})
    g['periodo'] = pd.Categorical([dd['periodo'].mode()[0]] * len(x), categories=dd['periodo'].cat.categories)
    g['fam_dens_w5_c'] = x - mA; g['fam2'] = g['fam_dens_w5_c'] ** 2; pr = r.get_prediction(g).summary_frame()
    ax.plot(x, pr['mean'], color=c, lw=1.8, label=f'{etq}; quadratic term p = {pc(r.pvalues["fam2"], 2)}'); ax.fill_between(x, pr['mean_ci_lower'], pr['mean_ci_upper'], color=c, alpha=.14, lw=0)
ax.set_ylabel('Predicted 3-year citations'); ax.legend(frameon=False, loc='lower center', fontsize=7.2); ax.set_ylim(0, None); letra(ax, 'A')
axh.hist(d257['fam_dens_w5'], bins=20, range=(0, 1), color=GR, edgecolor='white'); axh.set_yticks([]); axh.grid(False); axh.set_ylabel('Teams', fontsize=7.5)
axh.set_xlabel('Prior familiarity (density of tied pairs, previous 5 years)'); axh.spines['left'].set_visible(False); letra(axh, 'B', .8)
guarda(fig, 'F6_curva_familiaridad', 'Prior familiarity and expected citations')

# ---------- F7. Robustez ----------
var = [('multi_pais', 'Multinational\nteam', 1), ('conc_norm_c', 'Concentration\n(+1 SD)', sd['conc_norm_c']),
       ('div_campo_c', 'Field diversity\n(+1 SD)', sd['div_campo_c']), ('lider_any', 'Leader\npresent', 1), ('nuevos_share_c', 'New members\n(+1 SD)', sd['nuevos_share_c'])]
w95 = d257.assign(c3=d257['c3'].clip(upper=d257['c3'].quantile(.95))); sin5 = d257.sort_values('c3').iloc[:-5]
var_ = [('Full model', d257, m4, 'Poisson'), ('Citations capped (P95)', w95, m4, 'Poisson'), ('Excluding top 5 cited', sin5, m4, 'Poisson'),
        ('+ number of institutions', d241, m4 + ' + linst', 'Poisson'), ('Fully identified teams only', d212, m4.replace(' + cobertura_id', ''), 'Poisson'),
        ('OLS on log(1 + citations)', d257, m4, 'OLS')]
R = {}
for etq, dd, f, est in var_:
    r = ajusta(dd, f, est); ci = r.conf_int()
    for k, _, s in var: R[(etq, k)] = [np.exp(v * s) for v in (r.params[k], ci.loc[k, 0], ci.loc[k, 1])] + [r.pvalues[k]]
fig, axs = plt.subplots(1, 5, figsize=(7.4, 3.0), sharey=True, gridspec_kw={'wspace': .12})
for j, (ax, (k, t, _)) in enumerate(zip(axs, var)):
    for i, (etq, *_r) in enumerate(var_[::-1]):
        b, lo, hi, p = R[(etq, k)]; c = AZ if etq == 'Full model' else GR
        ax.errorbar(b, i, xerr=[[b - lo], [hi - b]], fmt='o', color=c, mfc=c if p < .05 else 'white', mec=c, ms=5, capsize=2, lw=1.2)
    ax.set_xscale('log'); ax.axvline(1, color=TX, lw=.9); ax.set_title(t, fontsize=8); ax.minorticks_off()
    ax.set_xticks([.5, 1, 2]); ax.set_xticklabels(['0.5', '1', '2']); ax.set_xlim(.4, 3.2); letra(ax, 'ABCDE'[j])
axs[0].set_yticks(range(len(var_))); axs[0].set_yticklabels([v[0] for v in var_[::-1]], fontsize=7.5)
fig.text(.5, -.03, 'Ratio of expected citations (log scale; filled = p < 0.05)', ha='center', fontsize=8)
guarda(fig, 'F7_robustez', 'Robustness of team effects')
pd.DataFrame({f'{a} | {b}': v for (a, b), v in R.items()}, index=['razon', 'ic_inf', 'ic_sup', 'p']).T.round(3).to_csv(OUT + 'tabla_robustez_final.csv')

# ---------- Diagnóstico: ¿la curva de familiaridad depende de los equipos de dos miembros? ----------
nm = np.exp(d257['lt']).round().astype(int); d257['n_m'] = nm
print('\nTwo-member teams:', int((nm == 2).sum()), f'({(nm == 2).mean() * 100:.1f} %)')
print(d257.groupby(nm.clip(upper=5))['fam_dens_w5'].agg(['count', 'mean', lambda s: (s == 0).mean(), lambda s: (s == 1).mean()]).round(2).rename(columns={'<lambda_0>': 'p_cero', '<lambda_1>': 'p_uno'}))
for etq, dd in [('Teams with >= 3 members only', d257[nm >= 3])]:
    try:
        f = m4
        r = ajusta(dd, f, 'Poisson'); print(etq, f'(N={len(dd)}): linear familiarity {r.params["fam_dens_w5_c"]:.2f} (p={r.pvalues["fam_dens_w5_c"]:.2f}); quadratic {r.params["fam2"]:.2f} (p={r.pvalues["fam2"]:.2f})')
    except Exception as e: print(etq, 'no estimable:', str(e)[:80])
print(pd.read_csv(OUT + 'tabla_robustez_final.csv').to_string(index=False))

# ---------- Cifras para la introducción (corpus final) ----------
Q = M[['SR', c_year]].drop_duplicates('SR').merge(tam, on='SR', how='left').rename(columns={c_year: 'anio'})
Qa = Q.dropna(subset=['n_aut'])
print('\nIntro | articles in corpus:', len(Q), '| with author data:', len(Qa), '| with >= 2 authors:', int((Qa['n_aut'] >= 2).sum()), f'({(Qa["n_aut"] >= 2).mean() * 100:.1f} %)')
for a_, b_ in [(2000, 2010), (2018, 2025)]:
    q_ = Qa[Qa['anio'].between(a_, b_)]; print(f'Intro | mean authors {a_}-{b_}: {q_["n_aut"].mean():.2f} (n={len(q_)})')
per = E.groupby('PersonID')['SR'].nunique(); print('Intro | distinct persons:', len(per), '| in one article only:', int((per == 1).sum()), '| in >1 article:', int((per > 1).sum()))
print('\nDone: F2–F7 (PNG and PDF) and tabla_robustez_final.csv in', OUT)


**Celda U.** Modelo adicional con el número de instituciones (N = 241), con y sin control de la calidad de la revista.

In [ ]:
# ===== CELDA U: modelo con instituciones (N = 241), con la base definitiva =====
# Ejecutar en la misma sesión, después de R, T y la celda de figuras K3 (usa d257, d241, m4 y ajusta).
sin_rev = m4.replace(' + lif_w', '')
filas = []
for est in ['Poisson', 'OLS']:
    r = ajusta(d241, m4 + ' + linst', est)
    filas.append((f'{est} | con control de revista + instituciones', r.params['linst'], r.bse['linst'], r.pvalues['linst']))
    r0 = ajusta(d241, sin_rev + ' + linst', est)
    filas.append((f'{est} | SIN control de revista + instituciones', r0.params['linst'], r0.bse['linst'], r0.pvalues['linst']))
print(f'N = {len(d241)}')
print(pd.DataFrame(filas, columns=['Modelo (coeficiente de log instituciones)', 'coef', 'EE', 'p']).round(3).to_string(index=False))
r = ajusta(d241, m4 + ' + linst', 'Poisson')
for k in ['multi_pais', 'conc_norm_c', 'lif_w']:
    print(f'Poisson N={len(d241)} | {k}: coef {r.params[k]:.3f} (EE {r.bse[k]:.3f}, p={r.pvalues[k]:.3f})')


**Celda V.** Mide el poder explicativo de los modelos y prueba si las variables de equipo, en conjunto, aportan algo una vez controlada la revista.

In [ ]:
# ===== CELDA V: cuánto explican los modelos (R², pseudo-R²) y prueba conjunta del bloque de equipo =====
# Ejecutar en la misma sesión, después de R (usa B y ajusta). No modifica ningún archivo.
d257 = B[B['n_id'] >= 2].copy(); d257['conc_norm_c'] = d257['conc_norm'] - d257['conc_norm'].mean()
c0 = 'lt + lexp + lrefs + lif_w + revision + alfabetico + cobertura_id + periodo + una_base'
c0_sin_rev = c0.replace(' + lif_w', '')
m2 = c0 + ' + fam_dens_w5_c + fam2'; m3 = m2 + ' + multi_pais + div_campo_c'; m4 = m3 + ' + lider_any + conc_norm_c + nuevos_share_c'
m5 = m4 + ' + fam_dens_w5_c:multi_pais + fam_dens_w5_c:div_campo_c + lider_any:nuevos_share_c'
modelos = {'M0 solo controles SIN revista': c0_sin_rev, 'M1 controles + revista': c0, 'M2 + familiaridad': m2, 'M3 + diversidad': m3, 'M4 + liderazgo y nuevos': m4, 'M5 + interacciones': m5}
filas = []
for nom, f in modelos.items():
    d_ = d257.assign(lc3=np.log1p(d257['c3']))
    o = smf.ols('lc3 ~ ' + f, d_).fit()
    p = smf.glm('c3 ~ ' + f, d_, family=sm.families.Poisson()).fit()
    filas.append((nom, round(o.rsquared, 3), round(o.rsquared_adj, 3), round(1 - p.deviance / p.null_deviance, 3)))
print(f'N = {len(d257)}')
print(pd.DataFrame(filas, columns=['Modelo', 'R2 (MCO)', 'R2 ajustado (MCO)', 'pseudo-R2 (Poisson, devianza)']).to_string(index=False))
equipo = ['fam_dens_w5_c', 'fam2', 'multi_pais', 'div_campo_c', 'lider_any', 'conc_norm_c', 'nuevos_share_c']
print('\nPrueba conjunta (Wald, errores robustos): ¿las 7 variables de equipo, juntas, aportan algo al modelo M4?')
for est in ['OLS', 'Poisson']:
    r = ajusta(d257, m4, est); w = r.wald_test(', '.join(f'{v} = 0' for v in equipo), scalar=True)
    print(f'  {est}: chi2/F = {float(w.statistic):.2f}, gl = 7, p = {float(w.pvalue):.3f}')
pp = smf.glm('c3 ~ ' + m4, d257, family=sm.families.Poisson()).fit()
print(f'\nSobredispersión del modelo Poisson M4: chi2 de Pearson / gl = {pp.pearson_chi2 / pp.df_resid:.2f} (1 = sin sobredispersión)')
print(f'Citas a 3 años: media = {d257["c3"].mean():.2f}, varianza = {d257["c3"].var():.1f}, artículos con 0 citas = {(d257["c3"] == 0).sum()} ({(d257["c3"] == 0).mean() * 100:.1f} %)')
